In [ ]:
import json
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup


# ============================================================
# AYARLAR
# ============================================================

BASE_URL = "https://www.tombankhadi.com"
LIST_URL = "https://www.tombankhadi.com/hadi-krediler"

OUTPUT_PATH = "/content/tom_katilim_finansmanlar.json"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "tr-TR,tr;q=0.9,en;q=0.8",
}


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def clean_text(text):
    if not text:
        return ""

    return re.sub(r"\s+", " ", text).strip()


def unique_list(values):
    result = []

    for value in values:
        value = clean_text(value)

        if value and value not in result:
            result.append(value)

    return result


def get_soup(session, url):
    response = session.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    print(f"HTTP STATUS: {response.status_code}")

    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")


# ============================================================
# FİNANSMAN LİNKLERİNİ BUL
# ============================================================

def get_financing_links(session):
    print(f"Finansman ana sayfası: {LIST_URL}")

    soup = get_soup(session, LIST_URL)

    links = []

    for a in soup.find_all("a", href=True):
        href = clean_text(a.get("href"))

        if not href:
            continue

        full_url = urljoin(BASE_URL, href)

        if "/hadi-krediler/" not in full_url:
            continue

        normalized = full_url.split("#")[0]
        normalized = normalized.split("?")[0]
        normalized = normalized.rstrip("/")

        if normalized == LIST_URL.rstrip("/"):
            continue

        if normalized not in links:
            links.append(normalized)

    return links


# ============================================================
# ÜRÜN İÇERİĞİNİ ÇEK
# ============================================================

def extract_product_content(soup):

    h1 = soup.find("h1")

    if not h1:
        return "", [], ""

    product_name = clean_text(
        h1.get_text(" ", strip=True)
    )

    texts = []
    conditions = []

    for element in h1.find_all_next(
        ["h2", "h3", "h4", "p", "li"]
    ):

        text = clean_text(
            element.get_text(" ", strip=True)
        )

        if not text:
            continue

        # Footer / banka açıklaması başladığında dur
        if "Hadi bir T.O.M. Katılım Bankası" in text:
            break

        if "31 Mart 2023 tarihinde" in text:
            break

        texts.append(text)

        if element.name == "li":
            conditions.append(text)

    texts = unique_list(texts)
    conditions = unique_list(conditions)

    raw_text = "\n".join(texts)

    return product_name, conditions, raw_text


# ============================================================
# ALAN ÇIKARIMLARI
# ============================================================

def extract_kar_payi_orani(text):

    results = []

    patterns = [
        r"(?i)k[aâ]r\s+oran[ıi][^%\d]{0,40}(%\s*\d+(?:[.,]\d+)?)",
        r"(?i)k[aâ]r\s+pay[ıi]\s+oran[ıi][^%\d]{0,40}(%\s*\d+(?:[.,]\d+)?)",
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text
        )

        for match in matches:

            match = clean_text(match)

            if match not in results:
                results.append(match)

    return results


def extract_financing_amount(text):

    results = []

    patterns = [
        r"\d[\d.]*\s*TL\s+ile\s+\d[\d.]*\s*TL\s+arasında",
        r"\d[\d.]*\s*TL\s*-\s*\d[\d.]*\s*TL",
        r"\d[\d.]*\s*TL(?:'ye|'ye|ye)?\s+kadar",
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        results.extend(matches)

    return unique_list(results)


def extract_vade(text):

    results = []

    patterns = [
        r"\d+\s+aya\s+varan",
        r"\d+\s+ay\s+vade",
        r"\d+\s+aya\s+kadar",
        r"\d+\s+ila\s+\d+\s+gün",
        r"\d+\s+güne\s+varan",
        r"\d+\s+ay",
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        results.extend(matches)

    return unique_list(results)


def extract_taksit(text):

    results = []

    patterns = [
        r"(\d+)\s+taksit",
        r"(\d+)\s+aya\s+varan\s+taksit",
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:

            value = clean_text(
                str(match)
            )

            if value and value not in results:
                results.append(value)

    return results


def extract_masraf_bilgisi(text):

    results = []

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    keywords = [
        "masraf",
        "ücret",
        "komisyon",
        "tahsis ücreti",
    ]

    for sentence in sentences:

        lower = sentence.lower()

        if any(
            keyword in lower
            for keyword in keywords
        ):
            results.append(sentence)

    return unique_list(results)


def detect_currency(text):

    currencies = []

    if re.search(r"\bTL\b", text):
        currencies.append("TRY")

    if re.search(
        r"\bUSD\b|\bdolar\b",
        text,
        re.IGNORECASE
    ):
        currencies.append("USD")

    if re.search(
        r"\bEUR\b|\beuro\b",
        text,
        re.IGNORECASE
    ):
        currencies.append("EUR")

    return currencies


def detect_category(product_name, text):

    combined = (
        product_name + " " + text
    ).lower()

    if "veresiye" in combined:
        return "alışveriş"

    if "alışveriş" in combined:
        return "alışveriş"

    if "eğitim" in combined:
        return "eğitim"

    if "seyahat" in combined:
        return "seyahat"

    if "sağlık" in combined:
        return "sağlık"

    return "diğer"


# ============================================================
# TEK ÜRÜNÜ ÇEK
# ============================================================

def scrape_product(session, url):

    print()
    print(f"Çekiliyor: {url}")

    soup = get_soup(
        session,
        url
    )

    product_name, conditions, raw_text = (
        extract_product_content(soup)
    )

    if not product_name:
        print("UYARI: Ürün adı bulunamadı.")
        return None

    print(f"Ürün: {product_name}")

    record = {

        "banka": "T.O.M. Katılım Bankası",

        "kayit_turu": "finansman",

        "urun_adi": product_name,

        "urun_kategorisi": detect_category(
            product_name,
            raw_text
        ),

        "kar_payi_orani": extract_kar_payi_orani(
            raw_text
        ),

        "finansman_orani": [],

        "finansman_tutari": extract_financing_amount(
            raw_text
        ),

        "vade": extract_vade(
            raw_text
        ),

        "taksit_sayisi": extract_taksit(
            raw_text
        ),

        "masraf_bilgisi": extract_masraf_bilgisi(
            raw_text
        ),

        "kampanya_turu": "",

        "kampanya_avantaji": [],

        "kampanya_suresi": "",

        "hedef_kitle": [],

        "para_birimi": detect_currency(
            raw_text
        ),

        "kosullar": conditions,

        "kaynak_url": url,

        "ham_metin": raw_text,
    }

    return record


# ============================================================
# ÇALIŞTIR
# ============================================================

print("=" * 100)
print("TOM KATILIM - FINANSMAN SCRAPER")
print("=" * 100)

session = requests.Session()

records = []

try:

    financing_links = get_financing_links(
        session
    )

    print()
    print(
        f"Toplam ürün linki: "
        f"{len(financing_links)}"
    )

    for index, url in enumerate(
        financing_links,
        start=1
    ):

        print()
        print(
            f"[{index}/{len(financing_links)}]"
        )

        try:

            record = scrape_product(
                session,
                url
            )

            if record:
                records.append(record)

        except Exception as e:

            print(
                f"HATA: "
                f"{type(e).__name__}: {e}"
            )

finally:

    session.close()


# ============================================================
# JSON KAYDET
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2
    )


print()
print("=" * 100)
print("TAMAMLANDI")
print("=" * 100)

print(
    f"Bulunan ürün linki : "
    f"{len(financing_links)}"
)

print(
    f"Başarılı kayıt     : "
    f"{len(records)}"
)

print(
    f"Dosya              : "
    f"{OUTPUT_PATH}"
)

print("=" * 100)


# ============================================================
# İLK KAYDI EKRANDA GÖSTER
# ============================================================

if records:

    print()
    print("İLK KAYIT:")
    print()

    print(
        json.dumps(
            records[0],
            ensure_ascii=False,
            indent=2
        )
    )


# ============================================================
# DOSYAYI BİLGİSAYARA İNDİR
# ============================================================

from google.colab import files

files.download(
    OUTPUT_PATH
)

TOM KATILIM - FINANSMAN SCRAPER
Finansman ana sayfası: https://www.tombankhadi.com/hadi-krediler
HTTP STATUS: 200

Toplam ürün linki: 3

[1/3]

Çekiliyor: https://www.tombankhadi.com/hadi-krediler/veresiye-kredi
HTTP STATUS: 200
Ürün: Veresiye Kredi

[2/3]

Çekiliyor: https://www.tombankhadi.com/hadi-krediler/taksitli-alisveris-kredisi
HTTP STATUS: 200
Ürün: Taksitli Kredi

[3/3]

Çekiliyor: https://www.tombankhadi.com/hadi-krediler/magazadan-alisveris-kredisi
HTTP STATUS: 200
Ürün: Mağazadan Alışveriş Kredisi

TAMAMLANDI
Bulunan ürün linki : 3
Başarılı kayıt     : 3
Dosya              : /content/tom_katilim_finansmanlar.json

İLK KAYIT:

{
  "banka": "T.O.M. Katılım Bankası",
  "kayit_turu": "finansman",
  "urun_adi": "Veresiye Kredi",
  "urun_kategorisi": "alışveriş",
  "kar_payi_orani": [],
  "finansman_orani": [],
  "finansman_tutari": [],
  "vade": [
    "45 ila 75 gün",
    "2 ay"
  ],
  "taksit_sayisi": [],
  "masraf_bilgisi": [],
  "kampanya_turu": "",
  "kampanya_avantaji": []

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# TOM KATILIM / HADI - KAMPANYA SCRAPER V2
# GOOGLE COLAB - TEK HÜCRE
# ============================================================

# ------------------------------------------------------------
# KURULUMLAR
# ------------------------------------------------------------

!pip -q install -U selenium beautifulsoup4 requests

# Colab'deki problemli chromium-browser/snap yerine
# doğrudan Google Chrome Stable kuruyoruz.
!wget -q -O /tmp/google-chrome.deb \
https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

!apt-get -qq update
!apt-get -qq install -y /tmp/google-chrome.deb


# ============================================================
# IMPORTLAR
# ============================================================

import os
import re
import json
import time
import shutil

from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.parse import urljoin, urlparse

import requests

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options


# ============================================================
# AYARLAR
# ============================================================

BASE_URL = "https://www.tombankhadi.com"

LIST_URL = (
    "https://www.tombankhadi.com/"
    "hadi-kazan/kampanyalar"
)

OUTPUT_PATH = (
    "/content/"
    "tom_katilim_kampanyalar.json"
)

HEADERS = {

    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/126.0.0.0 "
        "Safari/537.36"
    ),

    "Accept-Language": (
        "tr-TR,tr;q=0.9,en;q=0.8"
    ),
}

ISTANBUL = ZoneInfo(
    "Europe/Istanbul"
)


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def now_iso():

    return datetime.now(
        ISTANBUL
    ).isoformat(
        timespec="seconds"
    )


def clean_text(text):

    return re.sub(
        r"\s+",
        " ",
        str(text or "")
    ).strip()


def normalize_url(url):

    if not url:
        return ""

    url = url.split("#")[0]
    url = url.split("?")[0]

    return url.rstrip("/")


def is_campaign_url(url):

    if not url:
        return False

    parsed = urlparse(
        url
    )

    if (
        "tombankhadi.com"
        not in parsed.netloc.lower()
    ):
        return False

    path = (
        parsed.path
        .lower()
        .rstrip("/")
    )

    return (
        path.startswith(
            "/kampanyalar/"
        )
        or path.startswith(
            "/cok-kazananlar-kulubu-kampanya/"
        )
        or path.startswith(
            "/tom-bank-kazananlar-kulubu/"
        )
    )


# ============================================================
# SELENIUM DRIVER
# ============================================================

def create_driver():

    chrome_binary = (
        shutil.which(
            "google-chrome"
        )
        or shutil.which(
            "google-chrome-stable"
        )
        or "/usr/bin/google-chrome"
    )

    print(
        "Chrome binary:",
        chrome_binary
    )

    if not os.path.exists(
        chrome_binary
    ):

        raise RuntimeError(
            "Google Chrome bulunamadı."
        )

    options = Options()

    options.binary_location = (
        chrome_binary
    )

    options.add_argument(
        "--headless"
    )

    options.add_argument(
        "--no-sandbox"
    )

    options.add_argument(
        "--disable-dev-shm-usage"
    )

    options.add_argument(
        "--disable-gpu"
    )

    options.add_argument(
        "--window-size=1920,1080"
    )

    options.add_argument(
        "--disable-notifications"
    )

    options.add_argument(
        "--disable-extensions"
    )

    options.add_argument(
        "--lang=tr-TR"
    )

    options.add_argument(
        "--remote-debugging-port=9222"
    )

    # Selenium Manager otomatik olarak
    # Chrome sürümüne uygun driver'ı bulur.
    driver = webdriver.Chrome(
        options=options
    )

    driver.set_page_load_timeout(
        60
    )

    return driver


# ============================================================
# HTML'DEN KAMPANYA LİNKLERİ
# ============================================================

def campaign_links_from_html(
    html
):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    links = []

    for a in soup.find_all(
        "a",
        href=True
    ):

        full_url = normalize_url(
            urljoin(
                BASE_URL,
                a.get("href")
            )
        )

        if not is_campaign_url(
            full_url
        ):
            continue

        if full_url not in links:

            links.append(
                full_url
            )

    return links


# ============================================================
# TÜM KAMPANYALARI KEŞFET
# ============================================================

def discover_campaign_links():

    print(
        f"Kampanya ana sayfası: "
        f"{LIST_URL}"
    )

    driver = create_driver()

    try:

        driver.get(
            LIST_URL
        )

        time.sleep(
            3
        )

        last_count = -1
        click_count = 0

        while True:

            links = (
                campaign_links_from_html(
                    driver.page_source
                )
            )

            current_count = len(
                links
            )

            print(
                "Şu an bulunan kampanya:",
                current_count
            )

            # ------------------------------------------
            # Daha fazla göster
            # button / a / role=button olabilir
            # ------------------------------------------

            candidates = (
                driver.find_elements(
                    By.XPATH,
                    (
                        "//*["
                        "self::button "
                        "or self::a "
                        "or @role='button'"
                        "]"
                        "[contains("
                        "normalize-space(.), "
                        "'Daha fazla göster'"
                        ")]"
                    )
                )
            )

            visible = [
                item
                for item in candidates
                if item.is_displayed()
            ]

            if not visible:

                print(
                    "Daha fazla göster "
                    "butonu kalmadı."
                )

                break

            button = visible[0]

            driver.execute_script(
                (
                    "arguments[0]."
                    "scrollIntoView("
                    "{block:'center'}"
                    ");"
                ),
                button
            )

            time.sleep(
                0.5
            )

            driver.execute_script(
                "arguments[0].click();",
                button
            )

            click_count += 1

            print(
                "Daha fazla göster "
                f"tıklandı: {click_count}"
            )

            time.sleep(
                1.5
            )

            new_count = len(
                campaign_links_from_html(
                    driver.page_source
                )
            )

            # Aynı sayıda kalırsa
            # sonsuz döngü koruması
            if (
                new_count <= current_count
                and
                current_count == last_count
            ):

                print(
                    "İki turdur yeni kampanya "
                    "gelmedi, durduruluyor."
                )

                break

            last_count = (
                current_count
            )

            if click_count >= 100:

                print(
                    "100 tıklama güvenlik "
                    "sınırına ulaşıldı."
                )

                break

        # ------------------------------------------
        # SON HTML
        # ------------------------------------------

        soup = BeautifulSoup(
            driver.page_source,
            "html.parser"
        )

        records = {}

        for a in soup.find_all(
            "a",
            href=True
        ):

            url = normalize_url(
                urljoin(
                    BASE_URL,
                    a.get("href")
                )
            )

            if not is_campaign_url(
                url
            ):
                continue

            listing_title = ""

            node = a

            # Kartın üstündeki heading'i ara
            for _ in range(8):

                node = node.parent

                if node is None:
                    break

                heading = node.find(
                    [
                        "h2",
                        "h3",
                        "h4",
                        "h5",
                    ]
                )

                if heading:

                    candidate = clean_text(
                        heading.get_text(
                            " ",
                            strip=True
                        )
                    )

                    if (
                        candidate
                        and
                        candidate.lower()
                        not in {
                            "kampanyalar",
                            "hadi kazan",
                            "kampanya detayı",
                        }
                    ):

                        listing_title = (
                            candidate
                        )

                        break

            # Bulamazsa link yazısını kullan
            if not listing_title:

                listing_title = clean_text(
                    a.get_text(
                        " ",
                        strip=True
                    )
                )

            if (
                not listing_title
                or
                listing_title.lower()
                == "kampanya detayı"
            ):

                listing_title = (
                    "Kampanya"
                )

            if url not in records:

                records[url] = {
                    "url": url,
                    "listing_title": (
                        listing_title
                    )
                }

        results = list(
            records.values()
        )

        print()
        print(
            "Toplam keşfedilen kampanya:",
            len(results)
        )

        return results

    finally:

        driver.quit()


# ============================================================
# TARİH ÇIKARIMI
# ============================================================

MONTHS = (
    "Ocak|Şubat|Mart|Nisan|Mayıs|"
    "Haziran|Temmuz|Ağustos|Eylül|"
    "Ekim|Kasım|Aralık"
)


def extract_campaign_date(
    lines,
    text
):

    # --------------------------------------------------------
    # 20.8.2026 – 31.8.2026
    # 20.08.2026 - 31.08.2026
    # --------------------------------------------------------

    match = re.search(
        (
            r"\b"
            r"\d{1,2}"
            r"[./]"
            r"\d{1,2}"
            r"[./]"
            r"\d{4}"
            r"\s*[-–—]\s*"
            r"\d{1,2}"
            r"[./]"
            r"\d{1,2}"
            r"[./]"
            r"\d{4}"
            r"\b"
        ),
        text,
        flags=re.IGNORECASE
    )

    if match:

        return clean_text(
            match.group(0)
        )

    # --------------------------------------------------------
    # 13 Mart - 22 Mart 2025
    # 1 Temmuz 2026 - 31 Ağustos 2026
    # --------------------------------------------------------

    match = re.search(
        (
            rf"\b"
            rf"\d{{1,2}}\s+"
            rf"(?:{MONTHS})"
            rf"(?:\s+\d{{4}})?"
            rf"\s*[-–—]\s*"
            rf"\d{{1,2}}\s+"
            rf"(?:{MONTHS})"
            rf"\s+\d{{4}}"
            rf"\b"
        ),
        text,
        flags=re.IGNORECASE
    )

    if match:

        return clean_text(
            match.group(0)
        )

    # --------------------------------------------------------
    # 31 Ağustos 2026'ya kadar
    # --------------------------------------------------------

    match = re.search(
        (
            rf"\b"
            rf"\d{{1,2}}\s+"
            rf"(?:{MONTHS})\s+"
            rf"\d{{4}}"
            rf"(?:'ya|'ye|’ya|’ye)?"
            rf"\s+kadar"
            rf"\b"
        ),
        text,
        flags=re.IGNORECASE
    )

    if match:

        return clean_text(
            match.group(0)
        )

    # --------------------------------------------------------
    # Kampanya Tarihleri başlığının hemen sonrası
    # --------------------------------------------------------

    for i, line in enumerate(
        lines
    ):

        normalized = (
            clean_text(line)
            .lower()
            .replace(":", "")
        )

        if normalized == (
            "kampanya tarihleri"
        ):

            for candidate in lines[
                i + 1:
                i + 5
            ]:

                candidate = clean_text(
                    candidate
                )

                if re.search(
                    r"\d",
                    candidate
                ):

                    return candidate

    return ""


def extract_end_date(
    date_text
):

    if not date_text:
        return ""

    value = clean_text(
        date_text
    )

    value = re.sub(
        (
            r"(?:"
            r"'ya|'ye|’ya|’ye"
            r")?"
            r"\s+kadar$"
        ),
        "",
        value,
        flags=re.IGNORECASE
    ).strip()

    parts = re.split(
        r"\s*[-–—]\s*",
        value
    )

    if len(parts) >= 2:

        return clean_text(
            parts[-1]
        )

    return value


# ============================================================
# DETAY SAYFASI HAM METNİ
# ============================================================

def extract_raw_content(
    soup,
    title
):

    lines = []

    raw_lines = soup.get_text(
        "\n",
        strip=True
    ).splitlines()

    for line in raw_lines:

        line = clean_text(
            line
        )

        if line:
            lines.append(
                line
            )

    # --------------------------------------------------------
    # H1 başlangıcı
    # --------------------------------------------------------

    start_index = None

    for i, line in enumerate(
        lines
    ):

        if clean_text(line) == clean_text(
            title
        ):

            start_index = i
            break

    # --------------------------------------------------------
    # Yedek başlangıç
    # --------------------------------------------------------

    if start_index is None:

        for i, line in enumerate(
            lines
        ):

            normalized = (
                line
                .lower()
                .replace(":", "")
            )

            if normalized in {
                "kampanya tarihleri",
                "kampanya bilgileri",
            }:

                start_index = i
                break

    if start_index is None:

        start_index = 0

    # --------------------------------------------------------
    # Bitiş
    # --------------------------------------------------------

    end_index = len(
        lines
    )

    stop_markers = [

        "İlginizi Çekebilir",

        (
            "Hadi bir T.O.M. "
            "Katılım Bankası "
            "Anonim Şirketi "
            "uygulamasıdır."
        ),

        (
            "Hadi bir T.O.M. "
            "Katılım Bankası A.Ş."
        ),
    ]

    for i in range(
        start_index + 1,
        len(lines)
    ):

        if any(
            marker.lower()
            in lines[i].lower()
            for marker
            in stop_markers
        ):

            end_index = i
            break

    content_lines = lines[
        start_index:
        end_index
    ]

    # Arka arkaya duplicate satır temizliği
    cleaned = []

    for line in content_lines:

        if (
            not cleaned
            or
            cleaned[-1] != line
        ):

            cleaned.append(
                line
            )

    return (
        "\n".join(cleaned),
        cleaned
    )


# ============================================================
# TEK KAMPANYA
# ============================================================

def scrape_campaign(
    session,
    discovered_item
):

    url = (
        discovered_item[
            "url"
        ]
    )

    listing_title = (
        discovered_item[
            "listing_title"
        ]
    )

    print()
    print(
        f"Çekiliyor: {url}"
    )

    response = session.get(
        url,
        headers=HEADERS,
        timeout=30,
        allow_redirects=True
    )

    print(
        "HTTP STATUS:",
        response.status_code
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.content,
        "html.parser"
    )

    # --------------------------------------------------------
    # KAMPANYA ADI
    # --------------------------------------------------------

    h1 = soup.find(
        "h1"
    )

    if h1:

        campaign_name = clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )

    else:

        campaign_name = (
            listing_title
        )

    print(
        "Kampanya:",
        campaign_name
    )

    # --------------------------------------------------------
    # HAM METİN
    # --------------------------------------------------------

    raw_text, lines = (
        extract_raw_content(
            soup,
            campaign_name
        )
    )

    # --------------------------------------------------------
    # TARİH
    # --------------------------------------------------------

    campaign_date_text = (
        extract_campaign_date(
            lines,
            raw_text
        )
    )

    end_date = (
        extract_end_date(
            campaign_date_text
        )
    )

    print(
        "Tarih:",
        campaign_date_text
        or "bulunamadı"
    )

    print(
        "Ham metin:",
        len(raw_text),
        "karakter"
    )

    # --------------------------------------------------------
    # RAW JSON
    # --------------------------------------------------------

    return {

        "kampanya_adi": (
            campaign_name
        ),

        "liste_kategorisi": (
            "Kampanyalar"
        ),

        "liste_durumu": (
            "aktif"
        ),

        "liste_bitis_tarihi": (
            end_date
        ),

        "kaynak_url": (
            url
        ),

        "final_url": normalize_url(
            response.url
        ),

        "http_status": (
            response.status_code
        ),

        "listing_text": (
            listing_title
        ),

        "ham_metin": (
            raw_text
        ),
    }


# ============================================================
# MAIN
# ============================================================

print(
    "=" * 110
)

print(
    "TOM KATILIM / HADI "
    "- CAMPAIGN SCRAPER V2"
)

print(
    "=" * 110
)

print(
    f"Liste URL: "
    f"{LIST_URL}"
)

print(
    f"Output: "
    f"{OUTPUT_PATH}"
)

print()


# ============================================================
# DISCOVERY
# ============================================================

discovery_time = (
    now_iso()
)

discovered_campaigns = (
    discover_campaign_links()
)


# ============================================================
# DETAY SAYFALARI
# ============================================================

session = requests.Session()

campaign_records = []

errors = []

try:

    total = len(
        discovered_campaigns
    )

    print()
    print(
        "=" * 110
    )

    print(
        "DETAY SAYFALARI ÇEKİLİYOR"
    )

    print(
        "=" * 110
    )

    for index, item in enumerate(
        discovered_campaigns,
        start=1
    ):

        print()
        print(
            f"[{index}/{total}]"
        )

        try:

            record = scrape_campaign(
                session,
                item
            )

            campaign_records.append(
                record
            )

        except Exception as e:

            error = {

                "url": (
                    item["url"]
                ),

                "hata_turu": (
                    type(e).__name__
                ),

                "hata": (
                    str(e)
                ),
            }

            errors.append(
                error
            )

            print(
                "HATA:",
                type(e).__name__,
                str(e)
            )

        time.sleep(
            0.3
        )

finally:

    session.close()


# ============================================================
# DUPLICATE KONTROL
# ============================================================

seen_urls = set()
duplicate_urls = []

seen_titles = set()
duplicate_titles = []


for campaign in campaign_records:

    url = campaign[
        "final_url"
    ]

    title = (
        campaign[
            "kampanya_adi"
        ]
        .lower()
        .strip()
    )

    if url in seen_urls:

        duplicate_urls.append(
            url
        )

    seen_urls.add(
        url
    )

    if title in seen_titles:

        duplicate_titles.append(
            campaign[
                "kampanya_adi"
            ]
        )

    seen_titles.add(
        title
    )


# ============================================================
# JSON
# ============================================================

output = {

    "banka": (
        "T.O.M. Katılım Bankası"
    ),

    "liste_url": (
        LIST_URL
    ),

    "scrape_zamani": (
        now_iso()
    ),

    "discovery_dosyasi": (
        "TOM/Hadi kampanya listesi "
        "(Selenium - Daha fazla göster)"
    ),

    "discovery_zamani": (
        discovery_time
    ),

    "beklenen_aktif_kampanya_sayisi": (
        len(
            discovered_campaigns
        )
    ),

    "kampanya_sayisi": (
        len(
            campaign_records
        )
    ),

    "kampanyalar": (
        campaign_records
    ),
}


# ============================================================
# KAYDET
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SONUÇ
# ============================================================

print()
print(
    "=" * 110
)

print(
    "TAMAMLANDI"
)

print(
    "=" * 110
)

print(
    "Keşfedilen kampanya :",
    len(
        discovered_campaigns
    )
)

print(
    "Başarılı kayıt      :",
    len(
        campaign_records
    )
)

print(
    "Hata                :",
    len(
        errors
    )
)

print(
    "Duplicate URL       :",
    len(
        duplicate_urls
    )
)

print(
    "Duplicate başlık    :",
    len(
        duplicate_titles
    )
)

print(
    "Dosya               :",
    OUTPUT_PATH
)

print(
    "=" * 110
)


# ============================================================
# KAMPANYALAR
# ============================================================

print()
print(
    "ÇEKİLEN KAMPANYALAR:"
)
print()

for i, campaign in enumerate(
    campaign_records,
    start=1
):

    print(
        f"[{i:02d}] "
        f"{campaign['kampanya_adi']}"
    )

    print(
        "     Bitiş:",
        campaign[
            "liste_bitis_tarihi"
        ]
        or "-"
    )

    print(
        "     HTTP :",
        campaign[
            "http_status"
        ]
    )


# ============================================================
# HATALAR
# ============================================================

if errors:

    print()
    print(
        "=" * 110
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 110
    )

    for error in errors:

        print(
            error
        )


# ============================================================
# İNDİR
# ============================================================

from google.colab import files

files.download(
    OUTPUT_PATH
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatk1.0-data.
(Reading database ... 118888 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import re
import unicodedata

from datetime import datetime
from zoneinfo import ZoneInfo
from urllib.parse import urlparse


# ============================================================
# AYARLAR
# ============================================================

INPUT_PATH = "/content/tom_katilim_kampanyalar.json"

OUTPUT_PATH = (
    "/content/"
    "tom_katilim_kampanyalar_clean.json"
)

ISTANBUL = ZoneInfo(
    "Europe/Istanbul"
)


# ============================================================
# TÜRKÇE AYLAR
# ============================================================

MONTHS = {
    "ocak": 1,

    "şubat": 2,
    "subat": 2,

    "mart": 3,

    "nisan": 4,

    "mayıs": 5,
    "mayis": 5,

    "haziran": 6,

    "temmuz": 7,

    "ağustos": 8,
    "agustos": 8,

    "eylül": 9,
    "eylul": 9,

    "ekim": 10,

    "kasım": 11,
    "kasim": 11,

    "aralık": 12,
    "aralik": 12,
}


# ============================================================
# YARDIMCI FONKSİYONLAR
# ============================================================

def clean_text(text):

    return re.sub(
        r"\s+",
        " ",
        str(text or "")
    ).strip()


def normalize_search(text):
    """
    GEÇMİŞ / GEÇMŞİ gibi ifadelerde
    Türkçe büyük-küçük harf problemlerini
    azaltmak için normalize eder.
    """

    text = unicodedata.normalize(
        "NFKD",
        str(text or "").casefold()
    )

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    return text


def get_slug(url):
    """
    /kampanyalar/abc
    /cok-kazananlar-kulubu-kampanya/abc

    İkisinde de sonuç:
    abc
    """

    if not url:
        return ""

    path = urlparse(
        url
    ).path.rstrip("/")

    if not path:
        return ""

    return path.split("/")[-1]


# ============================================================
# TARİH PARSE
# ============================================================

def parse_date_value(text):

    text = clean_text(
        text
    )

    if not text:
        return None

    # --------------------------------------------------------
    # 31.08.2026
    # 31/08/2026
    #
    # Birden fazla tarih varsa sonuncuyu alır.
    # --------------------------------------------------------

    numeric_dates = re.findall(
        (
            r"(\d{1,2})"
            r"[./]"
            r"(\d{1,2})"
            r"[./]"
            r"(\d{4})"
        ),
        text
    )

    if numeric_dates:

        day, month, year = (
            numeric_dates[-1]
        )

        try:

            return datetime(
                int(year),
                int(month),
                int(day)
            ).date()

        except ValueError:

            pass

    # --------------------------------------------------------
    # 31 Ağustos 2026
    # 31 Ağustos2026
    # --------------------------------------------------------

    text_dates = re.findall(
        (
            r"(\d{1,2})"
            r"\s*"
            r"([A-Za-zÇĞİÖŞÜçğıöşü]+)"
            r"\s*"
            r"(\d{4})"
        ),
        text,
        flags=re.IGNORECASE
    )

    if text_dates:

        day, month_name, year = (
            text_dates[-1]
        )

        month = MONTHS.get(
            month_name.casefold()
        )

        if month:

            try:

                return datetime(
                    int(year),
                    month,
                    int(day)
                ).date()

            except ValueError:

                pass

    return None


# ============================================================
# HAM METİNDEN BİTİŞ TARİHİ
# ============================================================

def extract_end_date_from_raw(
    raw_text
):

    text = clean_text(
        raw_text
    )

    if not text:
        return None

    patterns = [

        # ----------------------------------------------------
        # Kampanya Tarihleri
        # 06/08/2026 - 31/08/2026
        # ----------------------------------------------------

        (
            r"kampanya\s+tarihleri"
            r".{0,100}?"
            r"("
            r"\d{1,2}"
            r"[./]"
            r"\d{1,2}"
            r"[./]"
            r"\d{4}"
            r")"
            r"\s*[-–—]\s*"
            r"("
            r"\d{1,2}"
            r"[./]"
            r"\d{1,2}"
            r"[./]"
            r"\d{4}"
            r")"
        ),

        # ----------------------------------------------------
        # Kampanya Tarihleri
        # 08 Ağustos - 31 Ağustos 2026
        # ----------------------------------------------------

        (
            r"kampanya\s+tarihleri"
            r".{0,100}?"
            r"("
            r"\d{1,2}"
            r"\s*"
            r"[A-Za-zÇĞİÖŞÜçğıöşü]+"
            r"(?:\s+\d{4})?"
            r")"
            r"\s*[-–—]\s*"
            r"("
            r"\d{1,2}"
            r"\s*"
            r"[A-Za-zÇĞİÖŞÜçğıöşü]+"
            r"\s*"
            r"\d{4}"
            r")"
        ),

        # ----------------------------------------------------
        # Kampanya 01.07.2026 - 31.08.2026
        # tarihleri arasında...
        # ----------------------------------------------------

        (
            r"kampanya"
            r".{0,220}?"
            r"("
            r"\d{1,2}"
            r"[./]"
            r"\d{1,2}"
            r"[./]"
            r"\d{4}"
            r")"
            r".{0,120}?"
            r"[-–—]"
            r".{0,120}?"
            r"("
            r"\d{1,2}"
            r"[./]"
            r"\d{1,2}"
            r"[./]"
            r"\d{4}"
            r")"
            r".{0,80}?"
            r"tarih"
        ),

        # ----------------------------------------------------
        # Kampanya 1 Temmuz 2026 -
        # 31 Ağustos2026 tarihleri arasında
        # ----------------------------------------------------

        (
            r"kampanya"
            r".{0,220}?"
            r"("
            r"\d{1,2}"
            r"\s*"
            r"[A-Za-zÇĞİÖŞÜçğıöşü]+"
            r"\s*"
            r"(?:\d{4})?"
            r")"
            r".{0,120}?"
            r"[-–—]"
            r".{0,120}?"
            r"("
            r"\d{1,2}"
            r"\s*"
            r"[A-Za-zÇĞİÖŞÜçğıöşü]+"
            r"\s*"
            r"\d{4}"
            r")"
            r".{0,80}?"
            r"tarih"
        ),

        # ----------------------------------------------------
        # Kampanya 31 Aralık 2026
        # tarihine kadar geçerlidir
        # ----------------------------------------------------

        (
            r"kampanya"
            r".{0,120}?"
            r"("
            r"\d{1,2}"
            r"\s*"
            r"[A-Za-zÇĞİÖŞÜçğıöşü]+"
            r"\s*"
            r"\d{4}"
            r")"
            r"(?:'ya|'ye|’ya|’ye)?"
            r"\s+"
            r"(?:tarihine\s+)?"
            r"kadar"
        ),
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if not match:
            continue

        # Range ise son tarih,
        # tek tarih ise ilk tarih
        candidate = match.group(
            match.lastindex
        )

        parsed = parse_date_value(
            candidate
        )

        if parsed:

            return parsed

    return None


# ============================================================
# JSON OKU
# ============================================================

with open(
    INPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(
        f
    )


campaigns = data.get(
    "kampanyalar",
    []
)


print(
    "=" * 100
)

print(
    "TOM KATILIM - "
    "KAMPANYA TEMİZLEME"
)

print(
    "=" * 100
)

print(
    "Ham kayıt sayısı:",
    len(campaigns)
)


# ============================================================
# DUPLICATE TEMİZLE
#
# BAŞLIK İLE DEĞİL!
# URL SLUG İLE!
# ============================================================

unique_campaigns = {}

alternative_urls = {}


for campaign in campaigns:

    source_url = (
        campaign.get(
            "kaynak_url"
        )
        or campaign.get(
            "final_url"
        )
        or ""
    )

    slug = get_slug(
        source_url
    )

    if not slug:

        continue

    # --------------------------------------------------------
    # İlk defa görüyorsak kaydet
    # --------------------------------------------------------

    if slug not in unique_campaigns:

        unique_campaigns[
            slug
        ] = campaign

        alternative_urls[
            slug
        ] = []

        continue

    # --------------------------------------------------------
    # Aynı slug tekrar geldi.
    #
    # /kampanyalar/
    # URL'sini tercih ediyoruz.
    # --------------------------------------------------------

    existing = unique_campaigns[
        slug
    ]

    existing_url = (
        existing.get(
            "kaynak_url",
            ""
        )
    )

    new_url = (
        campaign.get(
            "kaynak_url",
            ""
        )
    )

    existing_is_main = (
        "/kampanyalar/"
        in existing_url
    )

    new_is_main = (
        "/kampanyalar/"
        in new_url
    )

    # Alternatif URL'yi sakla
    if (
        new_url
        and
        new_url != existing_url
    ):

        alternative_urls[
            slug
        ].append(
            new_url
        )

    # Yeni kayıt ana /kampanyalar/ URL'siyse
    # onu canonical yap.
    if (
        new_is_main
        and
        not existing_is_main
    ):

        if existing_url:

            alternative_urls[
                slug
            ].append(
                existing_url
            )

        unique_campaigns[
            slug
        ] = campaign


# ============================================================
# DURUM VE TARİH TEMİZLİĞİ
# ============================================================

today = datetime.now(
    ISTANBUL
).date()


cleaned_campaigns = []

active_count = 0
past_count = 0

no_date_count = 0


for slug, original_campaign in (
    unique_campaigns.items()
):

    # Orijinali değiştirmemek için copy
    campaign = dict(
        original_campaign
    )

    title = campaign.get(
        "kampanya_adi",
        ""
    )

    normalized_title = (
        normalize_search(
            title
        )
    )

    # --------------------------------------------------------
    # Önce mevcut liste_bitis_tarihi
    # --------------------------------------------------------

    end_date = parse_date_value(
        campaign.get(
            "liste_bitis_tarihi",
            ""
        )
    )

    # --------------------------------------------------------
    # Yoksa ham metinden bul
    # --------------------------------------------------------

    if end_date is None:

        end_date = (
            extract_end_date_from_raw(
                campaign.get(
                    "ham_metin",
                    ""
                )
            )
        )

    # --------------------------------------------------------
    # GEÇMİŞ KAMPANYA etiketi
    #
    # Ayrıca sitedeki
    # "GEÇMŞİ KAMPANYA"
    # yazım hatasını da yakala.
    # --------------------------------------------------------

    past_marker = (

        "gecmis kampanya"
        in normalized_title

        or

        "gecmsi kampanya"
        in normalized_title
    )

    # --------------------------------------------------------
    # DURUM BELİRLE
    # --------------------------------------------------------

    if past_marker:

        status = "geçmiş"

        status_source = (
            "başlık_geçmiş_etiketi"
        )

    elif (
        end_date is not None
        and
        end_date < today
    ):

        status = "geçmiş"

        status_source = (
            "bitiş_tarihi"
        )

    else:

        status = "aktif"

        if end_date is not None:

            status_source = (
                "bitiş_tarihi"
            )

        else:

            status_source = (
                "tarih_yok_"
                "geçmiş_etiketi_yok"
            )

            no_date_count += 1

    # --------------------------------------------------------
    # SAY
    # --------------------------------------------------------

    if status == "aktif":

        active_count += 1

    else:

        past_count += 1

    # --------------------------------------------------------
    # KAYDI GÜNCELLE
    # --------------------------------------------------------

    campaign[
        "liste_durumu"
    ] = status

    campaign[
        "bitis_tarihi_iso"
    ] = (
        end_date.isoformat()
        if end_date
        else ""
    )

    campaign[
        "durum_kaynagi"
    ] = status_source

    campaign[
        "canonical_slug"
    ] = slug

    # --------------------------------------------------------
    # ALTERNATİF DUPLICATE URL
    # --------------------------------------------------------

    canonical_url = campaign.get(
        "kaynak_url",
        ""
    )

    alt_urls = []

    for url in alternative_urls.get(
        slug,
        []
    ):

        if (
            url
            and
            url != canonical_url
            and
            url not in alt_urls
        ):

            alt_urls.append(
                url
            )

    campaign[
        "alternatif_url"
    ] = alt_urls

    cleaned_campaigns.append(
        campaign
    )


# ============================================================
# ÜST SEVİYE JSON
# ============================================================

output = {

    "banka": data.get(
        "banka",
        "T.O.M. Katılım Bankası"
    ),

    "liste_url": data.get(
        "liste_url",
        ""
    ),

    "scrape_zamani": data.get(
        "scrape_zamani",
        ""
    ),

    "temizleme_zamani": (
        datetime.now(
            ISTANBUL
        ).isoformat(
            timespec="seconds"
        )
    ),

    "orijinal_kampanya_sayisi": (
        len(campaigns)
    ),

    "duplicate_silinen_kayit": (
        len(campaigns)
        -
        len(cleaned_campaigns)
    ),

    "kampanya_sayisi": (
        len(cleaned_campaigns)
    ),

    "aktif_kampanya_sayisi": (
        active_count
    ),

    "gecmis_kampanya_sayisi": (
        past_count
    ),

    "tarih_bulunamayan_kampanya_sayisi": (
        no_date_count
    ),

    "kampanyalar": (
        cleaned_campaigns
    ),
}


# ============================================================
# KAYDET
# ============================================================

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# KONTROLLER
# ============================================================

print()
print(
    "=" * 100
)

print(
    "TEMİZLEME TAMAMLANDI"
)

print(
    "=" * 100
)

print(
    "Ham kayıt              :",
    len(campaigns)
)

print(
    "Benzersiz kampanya     :",
    len(cleaned_campaigns)
)

print(
    "Silinen duplicate      :",
    len(campaigns)
    -
    len(cleaned_campaigns)
)

print(
    "Aktif kampanya         :",
    active_count
)

print(
    "Geçmiş kampanya        :",
    past_count
)

print(
    "Tarih bulunamayan      :",
    no_date_count
)

print(
    "Dosya                  :",
    OUTPUT_PATH
)

print(
    "=" * 100
)


# ============================================================
# AYNI BAŞLIK AMA FARKLI SLUG KONTROLÜ
# ============================================================

title_groups = {}


for campaign in cleaned_campaigns:

    title_key = clean_text(
        campaign.get(
            "kampanya_adi",
            ""
        )
    ).casefold()

    title_groups.setdefault(
        title_key,
        []
    ).append(
        campaign
    )


same_title_different_campaign = [

    group

    for group
    in title_groups.values()

    if len(group) > 1
]


print()
print(
    "Aynı başlıklı fakat "
    "ayrı kampanya sayısı:",
    len(
        same_title_different_campaign
    )
)


for group in (
    same_title_different_campaign
):

    print()
    print(
        "AYNI BAŞLIK / "
        "FARKLI KAMPANYA:"
    )

    for campaign in group:

        print(
            "-",
            campaign[
                "canonical_slug"
            ],
            "|",
            campaign[
                "bitis_tarihi_iso"
            ]
        )


# ============================================================
# İLK 10 AKTİF KAMPANYA
# ============================================================

print()
print(
    "=" * 100
)

print(
    "İLK 10 AKTİF KAMPANYA"
)

print(
    "=" * 100
)


active_campaigns = [

    campaign

    for campaign
    in cleaned_campaigns

    if (
        campaign[
            "liste_durumu"
        ]
        == "aktif"
    )
]


for index, campaign in enumerate(
    active_campaigns[:10],
    start=1
):

    print(
        f"[{index:02d}] "
        f"{campaign['kampanya_adi']}"
    )

    print(
        "     Bitiş:",
        campaign[
            "bitis_tarihi_iso"
        ]
        or "tarih yok"
    )


# ============================================================
# DOSYAYI İNDİR
# ============================================================

from google.colab import files

files.download(
    OUTPUT_PATH
)

TOM KATILIM - KAMPANYA TEMİZLEME
Ham kayıt sayısı: 161

TEMİZLEME TAMAMLANDI
Ham kayıt              : 161
Benzersiz kampanya     : 81
Silinen duplicate      : 80
Aktif kampanya         : 53
Geçmiş kampanya        : 28
Tarih bulunamayan      : 15
Dosya                  : /content/tom_katilim_kampanyalar_clean.json

Aynı başlıklı fakat ayrı kampanya sayısı: 1

AYNI BAŞLIK / FARKLI KAMPANYA:
- a101-ekstrada-tum-cep-telefonlarina-pesin-fiyatina-3-taksit | 2025-03-16
- a101ekstrada-tum-cep-telefonlarina-pesin-fiyatina-3taksit | 2025-02-09

İLK 10 AKTİF KAMPANYA
[01] A101’de her alışverişte %3'e varan nakit iade!
     Bitiş: 2026-12-31
[02] Toplam 1500 TL hoş geldin hediyesi! TOM1500 koduyla müşterimiz ol, 1500 TL senin olsun!
     Bitiş: 2026-08-31
[03] Hadi Alışveriş Kredisi ile Klima, Süpürge ve Televizyonlarda Vade Farksız 12 Taksit!
     Bitiş: 2026-08-31
[04] Hadi Black Kredi Kartı ile Restoderm’de %30 İndirim!
     Bitiş: 2026-08-30
[05] A101’lerde süt ürünleri harcamalarında %50 Hedi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import re
import sys
from urllib.parse import urlparse

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

INPUT_FILE = (
    "/content/"
    "tom_katilim_kampanyalar_clean.json"
)

OUTPUT_FILE = (
    "/content/"
    "tom_katilim_kampanya_extracted.json"
)

BANK_NAME = (
    "T.O.M. Katılım Bankası"
)


# ============================================================
# FINAL SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(value):

    value = str(
        value or ""
    )

    replacements = {
        "’": "'",
        "‘": "'",
        "´": "'",
        "`": "'",
        "–": "-",
        "—": "-",
    }

    for old, new in replacements.items():

        value = value.replace(
            old,
            new
        )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    return value.strip()


def normalize_match(value):

    value = normalize_text(
        value
    )

    value = value.replace(
        "İ",
        "i"
    )

    value = value.replace(
        "I",
        "ı"
    )

    value = value.casefold()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize_financial_text(
    value
):

    value = normalize_text(
        value
    )

    # 10% -> %10
    value = re.sub(
        (
            r"(?<![%\d])"
            r"([0-9]+(?:[.,][0-9]+)?)"
            r"\s*%"
        ),
        r"%\1",
        value
    )

    # % 10 -> %10
    value = re.sub(
        (
            r"%\s+"
            r"([0-9]+(?:[.,][0-9]+)?)"
        ),
        r"%\1",
        value
    )

    # Türkçe yüzde ondalık standardı
    # %0.99 -> %0,99
    value = re.sub(
        r"%(\d+)\.(\d+)",
        r"%\1,\2",
        value
    )

    # ₺ -> TL
    value = value.replace(
        "₺",
        " TL"
    )

    # 1000TL -> 1000 TL
    value = re.sub(
        r"(\d)\s*TL\b",
        r"\1 TL",
        value,
        flags=re.IGNORECASE
    )

    return normalize_text(
        value
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = normalize_text(
            value
        )

        key = normalize_match(
            value
        )

        if (
            value
            and
            key not in seen
        ):

            seen.add(
                key
            )

            result.append(
                value
            )

    return result


# ============================================================
# URL
# ============================================================

def get_slug(url):

    try:

        parsed = urlparse(
            str(
                url or ""
            )
        )

    except Exception:

        return ""

    parts = [
        part
        for part
        in parsed.path.split("/")
        if part
    ]

    if not parts:
        return ""

    return parts[-1]


# ============================================================
# RAW TEXT
# ============================================================

HEADINGS = {
    "kampanya tarihleri",
    "kampanya bilgileri",
    "kampanya detayları",
    "kampanya kuralları",
    "kampanya koşulları",
    "kampanya şartları",
    "kampanyadan yararlanabilecek kişiler",
    "kampanyadan yararlanacak kişiler",
    "kampanyadan kimler yararlanabilir",
}


def content_lines(
    raw_text,
    title
):

    result = []

    for raw_line in str(
        raw_text or ""
    ).splitlines():

        line = normalize_text(
            raw_line
        )

        if not line:
            continue

        normalized = (
            normalize_match(
                line
            )
            .rstrip(":")
        )

        if normalized in {
            "hemen indir",
            "daha fazla göster",
        }:
            continue

        result.append(
            line
        )

    return unique(
        result
    )


# ============================================================
# SECTION EXTRACTION
# ============================================================

def extract_section(
    lines,
    start_headings
):

    result = []
    active = False

    for line in lines:

        normalized = (
            normalize_match(
                line
            )
            .rstrip(":")
        )

        if normalized in start_headings:

            active = True
            continue

        if (
            active
            and normalized in HEADINGS
        ):
            break

        if active:

            result.append(
                line
            )

    return result


def extract_info_section(
    lines
):

    return extract_section(
        lines,
        {
            "kampanya bilgileri"
        }
    )


def extract_target_section(
    lines
):

    return extract_section(
        lines,
        {
            (
                "kampanyadan "
                "yararlanabilecek kişiler"
            ),
            (
                "kampanyadan "
                "yararlanacak kişiler"
            ),
            (
                "kampanyadan "
                "kimler yararlanabilir"
            ),
        }
    )


# ============================================================
# CAMPAIGN CATEGORY
# ============================================================

def classify_campaign(
    title,
    raw_text
):

    title_n = normalize_match(
        title
    )

    raw_n = normalize_match(
        raw_text
    )

    # İlk bölüm daha önemli.
    text = (
        title_n
        + " "
        + raw_n[:1600]
    )

    # --------------------------------------------------------
    # YENİ MÜŞTERİ
    # --------------------------------------------------------

    new_customer_markers = [
        "ilk kez tom bank müşter",
        "ilk kez hadi müşter",
        "müşterimiz ol",
        "hoş geldin",
        "davet kodu",
    ]

    if any(
        marker in text
        for marker
        in new_customer_markers
    ):

        return (
            "Yeni Müşteri Kampanyaları",
            "Yeni Müşteri Kampanyası",
        )

    # --------------------------------------------------------
    # SİGORTA
    # --------------------------------------------------------

    if "sigorta" in title_n:

        return (
            "Sigorta Kampanyaları",
            "Sigorta Kampanyası",
        )

    # --------------------------------------------------------
    # FİNANSMAN
    # --------------------------------------------------------

    finance_markers = [
        "taksitli alışveriş kredisi",
        "mağazadan alışveriş kredisi",
        "alışveriş kredisi",
        "sağlık kredisi",
        "hadi taksitli kredi",
        "hadi veresiye",
        " veresiye",
    ]

    if any(
        marker in text
        for marker
        in finance_markers
    ):

        return (
            "Finansman Kampanyaları",
            "Finansman Kampanyası",
        )

    # --------------------------------------------------------
    # YATIRIM
    # --------------------------------------------------------

    investment_markers = [
        "altın",
        "gümüş",
        "kazandıran hesap",
        "yatırım",
    ]

    if any(
        marker in title_n
        for marker
        in investment_markers
    ):

        return (
            "Yatırım Kampanyaları",
            "Yatırım Kampanyası",
        )

    # --------------------------------------------------------
    # ÖZEL BANKACILIK
    # --------------------------------------------------------

    if (
        "özel bankacılık"
        in text
    ):

        return (
            "Özel Bankacılık Kampanyaları",
            "Özel Bankacılık Kampanyası",
        )

    # --------------------------------------------------------
    # KART
    # --------------------------------------------------------

    card_markers = [
        "kredi kartı",
        "banka kartı",
        "hadi kart",
        "black kart",
        "mastercard",
    ]

    if any(
        marker in text
        for marker
        in card_markers
    ):

        return (
            "Kart Kampanyaları",
            "Kart Kampanyası",
        )

    # --------------------------------------------------------
    # GENEL
    # --------------------------------------------------------

    return (
        "Alışveriş Kampanyaları",
        "Alışveriş Kampanyası",
    )


# ============================================================
# CAMPAIGN BENEFIT
# ============================================================

def extract_benefits(
    title,
    lines
):

    info_lines = (
        extract_info_section(
            lines
        )
    )

    candidates = (
        [title]
        + info_lines
        + lines[:12]
    )

    keywords = (
        "indirim",
        "nakit iade",
        " iade",
        "hediye bakiye",
        "hediye",
        "taksit",
        "bedava",
        "vade farksız",
        "avantaj",
        "%",
    )

    # Avantaj değil, koşul.
    bad_markers = (
        "faydalanmak için",
        "faydalanabilmek için",
        "gerekm",
        "zorunlu",
        "geçerli değildir",
        "dahil değildir",
        "iptal",
        "iade edil",
        "iade olması",
        "iade halinde",
        "saklı tutar",
        "yüklenebilmesi",
        "kullanılmayan",
        "vade farklı",
        "detaylı bilgi",
        "tıkla",
        "başvuru için",
        "müşteriler kampanyadan yararlanabilir",
        "kişiye özel",
    )

    result = []

    for line in candidates:

        normalized = normalize_match(
            line
        )

        if any(
            marker in normalized
            for marker
            in bad_markers
        ):
            continue

        has_keyword = any(
            keyword in normalized
            for keyword
            in keywords
        )

        has_multiplier = (
            re.search(
                (
                    r"\b"
                    r"\d+(?:[.,]\d+)?"
                    r"\s*kat\s+kazan"
                ),
                normalized
            )
            is not None
        )

        if (
            has_keyword
            or has_multiplier
        ):

            result.append(
                normalize_financial_text(
                    line
                )
            )

        if len(result) >= 8:
            break

    return unique(
        result
    )


# ============================================================
# TAKSİT
# ============================================================

def extract_installments(
    title,
    lines
):

    info_lines = (
        extract_info_section(
            lines
        )
    )

    candidates = (
        [title]
        + info_lines
    )

    # İlk bölüm kampanya açısından esas.
    for line in lines[:20]:

        normalized = normalize_match(
            line
        )

        # Bunlar yasal sınır.
        # Kampanya avantajı değildir.
        if any(
            marker in normalized
            for marker
            in (
                "yasal düzenlem",
                "vade kısıtlam",
                "sınırı uygulan",
                "yasal kısıtlam",
            )
        ):

            continue

        # Vade farklı seçenekleri alma.
        if (
            "vade farklı"
            in normalized
            and
            "vade farksız"
            not in normalized
        ):

            continue

        if any(
            marker in normalized
            for marker
            in (
                "vade farksız",
                "peşin fiyatına",
                "taksit sayısı",
                "taksit sayıları",
                "aya varan taksit",
                " taksit",
            )
        ):

            candidates.append(
                line
            )

    text = "\n".join(
        candidates
    )

    values = []

    # --------------------------------------------------------
    # 3 veya 6 taksit
    # --------------------------------------------------------

    for match in re.finditer(
        (
            r"\b(\d{1,2})"
            r"\s*(?:ve|veya)\s*"
            r"(\d{1,2})"
            r"\s*taksit"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.extend(
            [
                match.group(1),
                match.group(2),
            ]
        )

    # --------------------------------------------------------
    # 3-6-9-12 taksit
    # 3,6,9,12 taksit
    # --------------------------------------------------------

    for match in re.finditer(
        (
            r"("
            r"(?:"
            r"\d{1,2}"
            r"\s*[-,/]\s*"
            r")+"
            r"\d{1,2}"
            r")"
            r"\s*taksit"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.extend(
            re.findall(
                r"\d{1,2}",
                match.group(1)
            )
        )

    # --------------------------------------------------------
    # 3 taksit
    # 12 aya varan taksit
    # --------------------------------------------------------

    for match in re.finditer(
        (
            r"\b(\d{1,2})"
            r"\s*"
            r"(?:aya?\s+varan\s+)?"
            r"taksit\w*"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.append(
            match.group(1)
        )

    # --------------------------------------------------------
    # "taksit sayıları: 3-6-9-12"
    # --------------------------------------------------------

    for match in re.finditer(
        (
            r"taksit\s+sayılar[ıi]"
            r"[^0-9]{0,20}"
            r"("
            r"(?:"
            r"\d{1,2}"
            r"\s*[-,/]\s*"
            r")+"
            r"\d{1,2}"
            r")"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.extend(
            re.findall(
                r"\d{1,2}",
                match.group(1)
            )
        )

    numbers = set()

    for value in values:

        if not value.isdigit():
            continue

        number = int(
            value
        )

        if (
            1
            <= number
            <= 36
        ):

            numbers.add(
                number
            )

    return [
        str(number)
        for number
        in sorted(
            numbers
        )
    ]


# ============================================================
# KÂR PAYI / FİNANSMAN ORANI
# ============================================================

def extract_rates(
    lines
):

    kar_payi = []
    finansman = []

    for line in lines[:25]:

        normalized_line = (
            normalize_financial_text(
                line
            )
        )

        normalized = normalize_match(
            normalized_line
        )

        rates = re.findall(
            (
                r"%"
                r"([0-9]+"
                r"(?:[.,][0-9]+)?)"
            ),
            normalized_line
        )

        for rate in rates:

            value = (
                "%"
                + rate.replace(
                    ".",
                    ","
                )
            )

            # ------------------------------------------------
            # Explicit kâr payı
            # ------------------------------------------------

            if any(
                marker in normalized
                for marker
                in (
                    "kar oran",
                    "kâr oran",
                    "kar pay",
                    "kâr pay",
                )
            ):

                kar_payi.append(
                    value
                )

            # ------------------------------------------------
            # Explicit vade farkı / finansman oranı
            #
            # "vade farksız" tek başına burada oran üretmez.
            # ------------------------------------------------

            elif any(
                marker in normalized
                for marker
                in (
                    "vade fark",
                    "finansman oran",
                )
            ):

                finansman.append(
                    value
                )

    return (
        unique(
            kar_payi
        ),
        unique(
            finansman
        ),
    )


# ============================================================
# VADE
# ============================================================

def extract_maturity(
    category,
    title,
    lines
):

    if (
        category
        != "Finansman Kampanyaları"
    ):
        return []

    candidates = (
        [title]
        + extract_info_section(
            lines
        )
        + lines[:12]
    )

    result = []

    for line in candidates:

        normalized = normalize_match(
            line
        )

        # "vade farksız" kâr avantajı,
        # doğrudan vade değeri değildir.
        if any(
            marker in normalized
            for marker
            in (
                "vade farksız",
                "vade farklı",
                "vade kısıtlam",
            )
        ):

            continue

        for match in re.finditer(
            (
                r"\b"
                r"(\d{1,2})"
                r"\s*aya\s+varan"
            ),
            line,
            flags=re.IGNORECASE
        ):

            result.append(
                (
                    f"{match.group(1)} "
                    "aya varan"
                )
            )

        for match in re.finditer(
            (
                r"\b"
                r"(\d{1,2})"
                r"\s*ay\s+vade"
            ),
            line,
            flags=re.IGNORECASE
        ):

            result.append(
                (
                    f"{match.group(1)} ay"
                )
            )

    return unique(
        result
    )


# ============================================================
# MASRAF
# ============================================================

def extract_expenses(
    lines
):

    result = []

    markers = (
        "komisyon",
        "tahsis ücreti",
        "ücret alın",
        "ücret tahsil",
        "masraf alın",
        "masraf tahsil",
        "masrafsız",
    )

    for line in lines:

        normalized = normalize_match(
            line
        )

        if any(
            marker in normalized
            for marker
            in markers
        ):

            result.append(
                normalize_financial_text(
                    line
                )
            )

    return unique(
        result
    )


# ============================================================
# TARGET AUDIENCE
# ============================================================

def extract_targets(
    raw_text,
    lines
):

    # Önce özel section varsa direkt al.
    result = extract_target_section(
        lines
    )

    normalized = normalize_match(
        raw_text
    )

    rules = [
        (
            (
                r"hadi\s+taksitli\s+kredi\s+"
                r"limiti[^.]{0,120}?"
                r"yeterli\s+olan\s+müşter"
            ),
            (
                "Hadi Taksitli Kredi limiti "
                "yeterli olan müşteriler"
            ),
        ),

        (
            (
                r"kampanya[^.]{0,100}?"
                r"yalnızca\s+hadi\s+black\s+"
                r"kredi\s+kart"
            ),
            (
                "Hadi Black Kredi Kartı "
                "sahipleri"
            ),
        ),

        (
            (
                r"kampanya[^.]{0,100}?"
                r"hadi\s+black\s+kredi\s+"
                r"kartı\s+ile"
            ),
            (
                "Hadi Black Kredi Kartı "
                "sahipleri"
            ),
        ),

        (
            (
                r"kampanya[^.]{0,100}?"
                r"yalnızca\s+hadi\s+kredi\s+"
                r"kartlar"
            ),
            (
                "Hadi Kredi Kartı sahipleri"
            ),
        ),

        (
            (
                r"kampanyadan\s+faydalanmak\s+"
                r"için\s+hadi\s+gold"
            ),
            "Hadi Gold üyeleri",
        ),

        (
            (
                r"çok\s+kazananlar\s+"
                r"kulüb[üu]\s+üyesi"
            ),
            (
                "Çok Kazananlar Kulübü "
                "üyeleri"
            ),
        ),

        (
            (
                r"özel\s+bankacılık\s+"
                r"classic.*?"
                r"elite.*?"
                r"prestige"
            ),
            (
                "TOM Bank Hadi Özel Bankacılık "
                "Classic, Elite, Elite Plus "
                "ve Prestige segment müşterileri"
            ),
        ),
    ]

    for pattern, label in rules:

        if re.search(
            pattern,
            normalized,
            flags=re.IGNORECASE
        ):

            result.append(
                label
            )

    if (
        "ilk kez tom bank"
        in normalized
        or
        "ilk kez hadi müşter"
        in normalized
    ):

        result.append(
            (
                "İlk kez TOM Bank/Hadi "
                "müşterisi olacak kişiler"
            )
        )

    if (
        "kişiye özel"
        in normalized
        or
        "seçili kitle"
        in normalized
    ):

        result.append(
            "Seçili müşteri kitlesi"
        )

    return unique(
        result
    )[:8]


# ============================================================
# PARA BİRİMİ
# ============================================================

def extract_currency(
    lines
):

    # Kampanyanın asıl bölümüne bak.
    text = "\n".join(
        lines[:25]
    )

    if re.search(
        (
            r"(?:"
            r"\bTL\b"
            r"|₺"
            r"|Türk\s+Liras"
            r")"
        ),
        text,
        flags=re.IGNORECASE
    ):

        return [
            "TL"
        ]

    return []


# ============================================================
# KOŞULLAR
# ============================================================

def extract_conditions(
    lines,
    title
):

    keywords = (
        "geçerli",
        "gerekm",
        "zorunlu",
        "dahil",
        "hariç",
        "en fazla",
        "en az",
        "minimum",
        "maksimum",
        "sadece",
        "yalnızca",
        "faydalan",
        "kullan",
        "seçil",
        "iade",
        "iptal",
        "sınırlı",
        "müşteri",
        "ödeme",
        "kod",
        "işlem",
        "alışveriş",
        "kredi kartı",
        "kredi",
        "veresiye",
        "vade",
        "limit",
        "katılım",
    )

    title_n = normalize_match(
        title
    )

    result = []

    for line in lines:

        normalized = (
            normalize_match(
                line
            )
            .rstrip(":")
        )

        # Başlığı koşullara tekrar alma.
        if (
            normalize_match(
                line
            )
            == title_n
        ):
            continue

        # Section heading alma.
        if normalized in HEADINGS:
            continue

        if (
            len(line) < 45
            and line.endswith(":")
        ):
            continue

        if any(
            keyword in normalized
            for keyword
            in keywords
        ):

            result.append(
                normalize_financial_text(
                    line
                )
            )

    return unique(
        result
    )


# ============================================================
# EXTRACT RECORD
# ============================================================

def extract_record(
    raw_record
):

    title = normalize_text(
        raw_record.get(
            "kampanya_adi",
            ""
        )
    )

    url = normalize_text(
        raw_record.get(
            "kaynak_url",
            ""
        )
    )

    raw_text_original = (
        raw_record.get(
            "ham_metin",
            ""
        )
    )

    raw_text = normalize_text(
        raw_text_original
    )

    lines = content_lines(
        raw_text_original,
        title
    )

    (
        category,
        campaign_type,
    ) = classify_campaign(
        title,
        raw_text
    )

    (
        kar_payi_orani,
        finansman_orani,
    ) = extract_rates(
        lines
    )

    # Clean dosyadaki normalize tarih
    # öncelikli.
    campaign_end = normalize_text(
        raw_record.get(
            "bitis_tarihi_iso",
            ""
        )
    )

    if not campaign_end:

        campaign_end = normalize_text(
            raw_record.get(
                "liste_bitis_tarihi",
                ""
            )
        )

    record = {

        "banka":
            BANK_NAME,

        "kayit_turu":
            "kampanya",

        "urun_adi":
            title,

        "urun_kategorisi":
            category,

        "kar_payi_orani":
            kar_payi_orani,

        "finansman_orani":
            finansman_orani,

        # Kampanya harcama eşikleri
        # finansman tutarı değildir.
        # Açık bir kredi miktarı yoksa boş.
        "finansman_tutari":
            [],

        "vade":
            extract_maturity(
                category,
                title,
                lines
            ),

        "taksit_sayisi":
            extract_installments(
                title,
                lines
            ),

        "masraf_bilgisi":
            extract_expenses(
                lines
            ),

        "kampanya_turu":
            campaign_type,

        "kampanya_avantaji":
            extract_benefits(
                title,
                lines
            ),

        "kampanya_suresi":
            campaign_end,

        "hedef_kitle":
            extract_targets(
                raw_text,
                lines
            ),

        "para_birimi":
            extract_currency(
                lines
            ),

        "kosullar":
            extract_conditions(
                lines,
                title
            ),

        "kaynak_url":
            url,

        "ham_metin":
            raw_text,
    }

    return record


# ============================================================
# SCHEMA VALIDATION
# ============================================================

def validate_schema(
    record,
    index
):

    errors = []

    if (
        list(
            record.keys()
        )
        != SCHEMA_KEYS
    ):

        errors.append(
            (
                f"[{index}] "
                "schema key/order uyuşmuyor."
            )
        )

    for field in LIST_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            list
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} list değil."
                )
            )

    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            str
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} string değil."
                )
            )

    if (
        record.get(
            "kayit_turu"
        )
        != "kampanya"
    ):

        errors.append(
            (
                f"[{index}] "
                "kayit_turu kampanya değil."
            )
        )

    serialized = json.dumps(
        record,
        ensure_ascii=False
    )

    # Ortak şemada TRY kullanmıyoruz.
    if "TRY" in serialized:

        errors.append(
            f"[{index}] TRY bulundu."
        )

    currencies = record.get(
        "para_birimi",
        []
    )

    if any(
        currency != "TL"
        for currency
        in currencies
    ):

        errors.append(
            (
                f"[{index}] "
                "desteklenmeyen para birimi."
            )
        )

    if (
        not record.get(
            "urun_adi"
        )
        or
        not record.get(
            "kaynak_url"
        )
        or
        not record.get(
            "ham_metin"
        )
    ):

        errors.append(
            (
                f"[{index}] "
                "zorunlu provenance alanı boş."
            )
        )

    return errors


# ============================================================
# SEMANTIC VALIDATION
# ============================================================

def contains(
    values,
    fragment
):

    fragment = normalize_match(
        fragment
    )

    return any(
        fragment
        in normalize_match(
            value
        )
        for value
        in values
    )


def semantic_validation(
    records
):

    errors = []

    by_slug = {
        get_slug(
            record[
                "kaynak_url"
            ]
        ):
        record
        for record
        in records
    }

    # --------------------------------------------------------
    # A101 %3
    # --------------------------------------------------------

    slug = (
        "a101de-her-alisveriste-"
        "3-nakit-iade"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "%3"
        ):

            errors.append(
                (
                    "A101 %3 -> "
                    "%3 avantajı çıkarılmadı."
                )
            )

        if (
            item[
                "kampanya_suresi"
            ]
            != "2026-12-31"
        ):

            errors.append(
                (
                    "A101 %3 -> "
                    "bitiş tarihi yanlış: "
                    f"{item['kampanya_suresi']}"
                )
            )

    # --------------------------------------------------------
    # TOM1500
    # --------------------------------------------------------

    slug = (
        "toplam-1500-tl-hos-geldin-"
        "hediyesi-tom1500-koduyla-"
        "musterimiz-ol-1500-tl-"
        "senin-olsun"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "1.500 TL"
        ):

            errors.append(
                (
                    "TOM1500 -> "
                    "1.500 TL avantajı "
                    "çıkarılmadı."
                )
            )

        if not any(
            "ilk kez"
            in normalize_match(
                target
            )
            for target
            in item[
                "hedef_kitle"
            ]
        ):

            errors.append(
                (
                    "TOM1500 -> "
                    "yeni müşteri hedef "
                    "kitlesi çıkarılmadı."
                )
            )

    # --------------------------------------------------------
    # KLİMA / SÜPÜRGE / TV
    # --------------------------------------------------------

    slug = (
        "hadi-alisveris-kredisi-ile-"
        "klima-supurge-ve-"
        "televizyonlarda-vade-"
        "farksiz-12-taksit"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if "12" not in item[
            "taksit_sayisi"
        ]:

            errors.append(
                (
                    "Klima kampanyası -> "
                    "12 taksit çıkarılmadı."
                )
            )

        # "vade farksız" diye
        # kar payına %0 yazılmamalı.
        if item[
            "kar_payi_orani"
        ]:

            errors.append(
                (
                    "Klima kampanyası -> "
                    "vade farksız ifadesinden "
                    "hatalı kâr payı üretildi."
                )
            )

    # --------------------------------------------------------
    # RESTODERM
    # --------------------------------------------------------

    slug = (
        "hadi-black-karti-ile-"
        "restoderm-alisverislerinde-"
        "30-indirim"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "%30"
        ):

            errors.append(
                (
                    "Restoderm -> "
                    "%30 avantajı çıkarılmadı."
                )
            )

    # --------------------------------------------------------
    # SÜT %50
    # --------------------------------------------------------

    slug = (
        "a101lerde-sut-urunleri-"
        "harcamalarinda-50-"
        "hediye-bakiye-kazan"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "%50"
        ):

            errors.append(
                (
                    "Süt kampanyası -> "
                    "%50 çıkarılmadı."
                )
            )

    # --------------------------------------------------------
    # MTV 3 TAKSİT
    # --------------------------------------------------------

    slug = (
        "vergi-kampanyasi"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if (
            item[
                "taksit_sayisi"
            ]
            != [
                "3"
            ]
        ):

            errors.append(
                (
                    "MTV -> taksit yanlış: "
                    f"{item['taksit_sayisi']}"
                )
            )

    # --------------------------------------------------------
    # GİYİM 3 / 6
    # --------------------------------------------------------

    slug = (
        "giyim-alisverislerinde-"
        "kampanya"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if (
            item[
                "taksit_sayisi"
            ]
            != [
                "3",
                "6",
            ]
        ):

            errors.append(
                (
                    "Giyim -> taksit yanlış: "
                    f"{item['taksit_sayisi']}"
                )
            )

    # --------------------------------------------------------
    # ECZANE %0 / 3-6 TAKSİT
    # --------------------------------------------------------

    slug = (
        "eczanelerde-hadi-saglik-"
        "kredisi-ile-0-vade-farki-"
        "ile-6-taksit"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if (
            item[
                "taksit_sayisi"
            ]
            != [
                "3",
                "6",
            ]
        ):

            errors.append(
                (
                    "Eczane -> taksit yanlış: "
                    f"{item['taksit_sayisi']}"
                )
            )

        if not contains(
            item[
                "finansman_orani"
            ],
            "%0"
        ):

            errors.append(
                (
                    "Eczane -> explicit "
                    "%0 vade farkı çıkarılmadı."
                )
            )

        # %0 finansman oranıdır,
        # kâr payı alanına gitmemeli.
        if item[
            "kar_payi_orani"
        ]:

            errors.append(
                (
                    "Eczane -> %0 yanlışlıkla "
                    "kâr payına yazıldı."
                )
            )

    # --------------------------------------------------------
    # SAMSUNG CEP TELEFONU
    #
    # Yasal 12 taksit sınırını
    # kampanya avantajı sanma.
    # --------------------------------------------------------

    slug = (
        "tom-bank-hadi-taksitli-"
        "alisveris-kredisi-ile-"
        "samsung-cep-telefonlarinda-"
        "3-taksit-firsati"
    )

    if slug in by_slug:

        item = by_slug[
            slug
        ]

        if (
            item[
                "taksit_sayisi"
            ]
            != [
                "3"
            ]
        ):

            errors.append(
                (
                    "Samsung cep telefonu -> "
                    "kampanya taksiti 3 olmalı: "
                    f"{item['taksit_sayisi']}"
                )
            )

    return errors


# ============================================================
# LOAD
# ============================================================

print(
    "=" * 110
)

print(
    "TOM KATILIM - "
    "CAMPAIGN EXTRACTOR V1"
)

print(
    "=" * 110
)

print(
    "Input :",
    INPUT_FILE
)

print(
    "Output:",
    OUTPUT_FILE
)

print()


with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as file:

    raw_data = json.load(
        file
    )


raw_records = raw_data.get(
    "kampanyalar",
    []
)


if not isinstance(
    raw_records,
    list
):

    raise TypeError(
        "kampanyalar alanı list değil."
    )


EXPECTED_COUNT = raw_data.get(
    "aktif_kampanya_sayisi",
    53
)


# ============================================================
# SADECE AKTİF KAMPANYALAR
# ============================================================

active_records = [
    record
    for record
    in raw_records
    if (
        normalize_match(
            record.get(
                "liste_durumu",
                ""
            )
        )
        == "aktif"
    )
]


# ============================================================
# EXTRACTION
# ============================================================

extracted = []

extraction_errors = []

schema_errors = []


for index, raw_record in enumerate(
    active_records,
    start=1
):

    try:

        record = extract_record(
            raw_record
        )

        extracted.append(
            record
        )

        errors = validate_schema(
            record,
            index
        )

        schema_errors.extend(
            errors
        )

    except Exception as error:

        extraction_errors.append(
            (
                f"[{index}] "
                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ============================================================
# DUPLICATE URL
# ============================================================

urls = [
    record[
        "kaynak_url"
    ]
    for record
    in extracted
]


duplicate_url_count = (
    len(urls)
    -
    len(
        set(
            urls
        )
    )
)


# ============================================================
# SEMANTIC VALIDATION
# ============================================================

semantic_errors = (
    semantic_validation(
        extracted
    )
)


# ============================================================
# COUNT VALIDATION
# ============================================================

if (
    len(active_records)
    != EXPECTED_COUNT
):

    semantic_errors.append(
        (
            "Aktif RAW kayıt sayısı "
            "beklenenle uyuşmuyor: "
            f"{len(active_records)} "
            f"!= {EXPECTED_COUNT}"
        )
    )


if (
    len(extracted)
    != EXPECTED_COUNT
):

    semantic_errors.append(
        (
            "Extracted kayıt sayısı "
            "beklenenle uyuşmuyor: "
            f"{len(extracted)} "
            f"!= {EXPECTED_COUNT}"
        )
    )


if duplicate_url_count != 0:

    semantic_errors.append(
        (
            "Duplicate kaynak URL var: "
            f"{duplicate_url_count}"
        )
    )


# ============================================================
# SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        extracted,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# RESULT
# ============================================================

total_errors = (
    len(
        extraction_errors
    )
    +
    len(
        schema_errors
    )
    +
    len(
        semantic_errors
    )
)


print()
print(
    "=" * 110
)

print(
    "CAMPAIGN EXTRACTION V1 SONUCU"
)

print(
    "=" * 110
)

print(
    "Beklenen aktif :",
    EXPECTED_COUNT
)

print(
    "Clean RAW       :",
    len(
        raw_records
    )
)

print(
    "Aktif RAW       :",
    len(
        active_records
    )
)

print(
    "Extracted       :",
    len(
        extracted
    )
)

print(
    "Duplicate URL   :",
    duplicate_url_count
)

print(
    "Extraction error:",
    len(
        extraction_errors
    )
)

print(
    "Schema error    :",
    len(
        schema_errors
    )
)

print(
    "Semantic error  :",
    len(
        semantic_errors
    )
)

print(
    "Toplam error    :",
    total_errors
)


# ============================================================
# ERROR DETAILS
# ============================================================

if extraction_errors:

    print()
    print(
        "EXTRACTION HATALARI:"
    )

    for error in extraction_errors:

        print(
            "-",
            error
        )


if schema_errors:

    print()
    print(
        "SCHEMA HATALARI:"
    )

    for error in schema_errors:

        print(
            "-",
            error
        )


if semantic_errors:

    print()
    print(
        "SEMANTİK HATALAR:"
    )

    for error in semantic_errors:

        print(
            "-",
            error
        )


# ============================================================
# CATEGORY COUNTS
# ============================================================

print()
print(
    "=" * 110
)

print(
    "KATEGORİ DAĞILIMI"
)

print(
    "=" * 110
)


categories = {}


for record in extracted:

    category = record[
        "urun_kategorisi"
    ]

    categories[
        category
    ] = (
        categories.get(
            category,
            0
        )
        + 1
    )


for category, count in sorted(
    categories.items()
):

    print(
        f"{category}: {count}"
    )


# ============================================================
# FIRST RECORD
# ============================================================

if extracted:

    print()
    print(
        "=" * 110
    )

    print(
        "İLK PROCESSED KAYIT"
    )

    print(
        "=" * 110
    )

    print(
        json.dumps(
            extracted[0],
            ensure_ascii=False,
            indent=2
        )
    )


# ============================================================
# FINAL
# ============================================================

print()
print(
    "=" * 110
)

if total_errors == 0:

    print(
        "SONUÇ: TOM KATILIM "
        "CAMPAIGN EXTRACTION V1 "
        "BAŞARILI ✅"
    )

    print(
        (
            f"{len(extracted)} aktif kampanya "
            "18-key final schema'ya "
            "başarıyla dönüştürüldü ✅"
        )
    )

else:

    print(
        "SONUÇ: TOM KATILIM "
        "CAMPAIGN EXTRACTION V1 "
        "KONTROL GEREKİYOR ❌"
    )


print()
print(
    "JSON:",
    OUTPUT_FILE
)

print(
    "=" * 110
)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    OUTPUT_FILE
)

TOM KATILIM - CAMPAIGN EXTRACTOR V1
Input : /content/tom_katilim_kampanyalar_clean.json
Output: /content/tom_katilim_kampanya_extracted.json


CAMPAIGN EXTRACTION V1 SONUCU
Beklenen aktif : 53
Clean RAW       : 81
Aktif RAW       : 53
Extracted       : 53
Duplicate URL   : 0
Extraction error: 0
Schema error    : 0
Semantic error  : 0
Toplam error    : 0

KATEGORİ DAĞILIMI
Finansman Kampanyaları: 27
Kart Kampanyaları: 21
Sigorta Kampanyaları: 1
Yeni Müşteri Kampanyaları: 2
Özel Bankacılık Kampanyaları: 2

İLK PROCESSED KAYIT
{
  "banka": "T.O.M. Katılım Bankası",
  "kayit_turu": "kampanya",
  "urun_adi": "A101'de her alışverişte %3'e varan nakit iade!",
  "urun_kategorisi": "Finansman Kampanyaları",
  "kar_payi_orani": [],
  "finansman_orani": [],
  "finansman_tutari": [],
  "vade": [],
  "taksit_sayisi": [],
  "masraf_bilgisi": [],
  "kampanya_turu": "Finansman Kampanyası",
  "kampanya_avantaji": [
    "A101'de her alışverişte %3'e varan nakit iade!",
    "Kampanya kapsamında bir takvi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import re
from urllib.parse import urlparse

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

INPUT_FILE = "/content/tom_katilim_kampanyalar_clean.json"
OUTPUT_FILE = "/content/tom_katilim_kampanya_extracted_v2.json"

BANK_NAME = "T.O.M. Katılım Bankası"


# ============================================================
# FINAL SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]

LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}

SCALAR_FIELDS = set(SCHEMA_KEYS) - LIST_FIELDS


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(value):

    value = str(value or "")

    replacements = {
        "’": "'",
        "‘": "'",
        "´": "'",
        "`": "'",
        "–": "-",
        "—": "-",
        "\xa0": " ",
    }

    for old, new in replacements.items():
        value = value.replace(old, new)

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    return value.strip()


def normalize_match(value):

    value = normalize_text(value)

    value = value.replace(
        "İ",
        "i"
    )

    value = value.replace(
        "I",
        "ı"
    )

    value = value.casefold()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize_financial_text(value):

    value = normalize_text(value)

    # 10% -> %10
    value = re.sub(
        r"(?<![%\d])([0-9]+(?:[.,][0-9]+)?)\s*%",
        r"%\1",
        value
    )

    # % 10 -> %10
    value = re.sub(
        r"%\s+([0-9]+(?:[.,][0-9]+)?)",
        r"%\1",
        value
    )

    # %0.99 -> %0,99
    value = re.sub(
        r"%(\d+)\.(\d+)",
        r"%\1,\2",
        value
    )

    value = value.replace(
        "₺",
        " TL"
    )

    value = re.sub(
        r"(\d)\s*TL\b",
        r"\1 TL",
        value,
        flags=re.IGNORECASE
    )

    return normalize_text(value)


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = normalize_text(value)

        if not value:
            continue

        key = normalize_match(value)

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


# ============================================================
# URL
# ============================================================

def get_slug(url):

    try:
        parsed = urlparse(
            str(url or "")
        )

    except Exception:
        return ""

    parts = [
        part
        for part in parsed.path.split("/")
        if part
    ]

    if not parts:
        return ""

    return parts[-1]


# ============================================================
# RAW TEXT
# ============================================================

SECTION_HEADINGS = {
    "kampanya tarihleri",
    "kampanya bilgileri",
    "kampanya detayları",
    "kampanya kuralları",
    "kampanya koşulları",
    "kampanya şartları",
    "kampanyadan yararlanabilecek kişiler",
    "kampanyadan yararlanacak kişiler",
    "kampanyadan kimler yararlanabilir",
}


def content_lines(raw_text, title):

    result = []

    for raw_line in str(
        raw_text or ""
    ).splitlines():

        line = normalize_text(raw_line)

        if not line:
            continue

        normalized = (
            normalize_match(line)
            .rstrip(":")
        )

        if normalized in {
            "hemen indir",
            "daha fazla göster",
        }:
            continue

        result.append(line)

    return unique(result)


# ============================================================
# SECTION
# ============================================================

def extract_section(
    lines,
    wanted_headings
):

    active = False
    result = []

    for line in lines:

        normalized = (
            normalize_match(line)
            .rstrip(":")
        )

        if normalized in wanted_headings:
            active = True
            continue

        if (
            active
            and normalized in SECTION_HEADINGS
        ):
            break

        if active:
            result.append(line)

    return result


def extract_info_section(lines):

    return extract_section(
        lines,
        {
            "kampanya bilgileri"
        }
    )


def extract_target_section(lines):

    return extract_section(
        lines,
        {
            "kampanyadan yararlanabilecek kişiler",
            "kampanyadan yararlanacak kişiler",
            "kampanyadan kimler yararlanabilir",
        }
    )


# ============================================================
# CATEGORY
# ============================================================

def classify_campaign(
    title,
    lines
):

    title_n = normalize_match(title)

    info_lines = extract_info_section(
        lines
    )

    info_text = normalize_match(
        " ".join(info_lines)
    )

    # İlk bölüm yalnızca kart/genel ayrımı için kullanılacak.
    early_text = normalize_match(
        " ".join(lines[:14])
    )

    # --------------------------------------------------------
    # YENİ MÜŞTERİ
    # --------------------------------------------------------

    new_customer_text = (
        title_n
        + " "
        + info_text
    )

    if any(
        marker in new_customer_text
        for marker in [
            "müşterimiz ol",
            "ilk kez tom bank",
            "ilk kez hadi müşter",
            "hoş geldin",
            "davet kodu",
        ]
    ):

        return (
            "Yeni Müşteri Kampanyaları",
            "Yeni Müşteri Kampanyası",
        )

    # --------------------------------------------------------
    # SİGORTA
    # --------------------------------------------------------

    if "sigorta" in title_n:

        return (
            "Sigorta Kampanyaları",
            "Sigorta Kampanyası",
        )

    # --------------------------------------------------------
    # FİNANSMAN
    #
    # Kritik fark:
    # bütün ham metne bakmıyoruz.
    # Başlık + Kampanya Bilgileri esas.
    # --------------------------------------------------------

    finance_primary = (
        title_n
        + " "
        + info_text
    )

    finance_markers = [
        "taksitli alışveriş kredisi",
        "mağazadan alışveriş kredisi",
        "alışveriş kredisi",
        "taksitli sağlık kredisi",
        "sağlık kredisi",
        "hadi taksitli kredi",
        "hadi veresiye",
        "veresiye kampanya",
    ]

    if any(
        marker in finance_primary
        for marker in finance_markers
    ):

        return (
            "Finansman Kampanyaları",
            "Finansman Kampanyası",
        )

    # --------------------------------------------------------
    # ÖZEL BANKACILIK
    # --------------------------------------------------------

    if (
        "özel bankacılık"
        in title_n
        or
        "özel bankacılık"
        in info_text
    ):

        return (
            "Özel Bankacılık Kampanyaları",
            "Özel Bankacılık Kampanyası",
        )

    # --------------------------------------------------------
    # YATIRIM
    # --------------------------------------------------------

    if any(
        marker in title_n
        for marker in [
            "altın",
            "gümüş",
            "yatırım",
            "kazandıran hesap",
        ]
    ):

        return (
            "Yatırım Kampanyaları",
            "Yatırım Kampanyası",
        )

    # --------------------------------------------------------
    # KART
    # --------------------------------------------------------

    card_markers = [
        "kredi kartı",
        "black kart",
        "hadi kart",
        "mastercard",
        "hesap kartı",
    ]

    if any(
        marker in (
            title_n
            + " "
            + info_text
            + " "
            + early_text
        )
        for marker in card_markers
    ):

        return (
            "Kart Kampanyaları",
            "Kart Kampanyası",
        )

    # --------------------------------------------------------
    # GENEL
    # --------------------------------------------------------

    return (
        "Alışveriş Kampanyaları",
        "Alışveriş Kampanyası",
    )


# ============================================================
# BENEFIT
# ============================================================

NEGATIVE_BENEFIT_MARKERS = (
    "kazanımı yok",
    "kazanım yok",
    "kazanılmaz",
    "kazanılamaz",
    "iade verilmeyecek",
    "iade verilmeyecektir",
    "geri alınır",
    "geri alınacaktır",
    "borç olarak yansıtılır",
    "yansıtılır",
    "yansıtılacaktır",
    "yüklenebilmesi için",
    "yükleme tarihinde",
    "kullanılmayan",
    "iptal edil",
    "iptal/iade",
    "iptal veya iade",
    "iade olması durum",
    "iade edilmesi durum",
    "geçerli değildir",
    "dahil değildir",
    "faydalanmak için",
    "faydalanabilmek için",
    "gerekmektedir",
    "gereklidir",
    "zorunludur",
    "saklı tutar",
)


def is_negative_benefit(line):

    normalized = normalize_match(
        line
    )

    return any(
        marker in normalized
        for marker
        in NEGATIVE_BENEFIT_MARKERS
    )


def extract_benefits(
    title,
    lines
):

    info_lines = extract_info_section(
        lines
    )

    candidates = (
        [title]
        + info_lines
        + lines[:18]
    )

    result = []

    keywords = (
        "indirim",
        "nakit iade",
        "hediye bakiye",
        "hediye",
        "bedava",
        "vade farksız",
        "peşin fiyatına",
        "taksit",
        "%",

        # "2 kat kazan"
        "kat kazan",
    )

    for line in candidates:

        normalized = normalize_match(
            line
        )

        if (
            normalized.rstrip(":")
            in SECTION_HEADINGS
        ):
            continue

        if is_negative_benefit(
            line
        ):
            continue

        if any(
            keyword in normalized
            for keyword in keywords
        ):

            result.append(
                normalize_financial_text(
                    line
                )
            )

        if len(result) >= 8:
            break

    return unique(
        result
    )


# ============================================================
# TAKSİT
# ============================================================

def extract_installments(
    title,
    lines
):

    info_lines = extract_info_section(
        lines
    )

    candidates = [
        title,
        *info_lines,
    ]

    for line in lines[:22]:

        normalized = normalize_match(
            line
        )

        # Yasal taksit sınırlarını kampanya avantajı sanma.
        if any(
            marker in normalized
            for marker in (
                "yasal düzenlem",
                "yasal kısıtlam",
                "vade kısıtlam",
                "sınırı uygulan",
            )
        ):
            continue

        if (
            "vade farklı"
            in normalized
            and
            "vade farksız"
            not in normalized
        ):
            continue

        if any(
            marker in normalized
            for marker in (
                "vade farksız",
                "peşin fiyatına",
                "taksit sayısı",
                "taksit sayıları",
                "aya varan taksit",
                " taksit",
            )
        ):
            candidates.append(
                line
            )

    text = "\n".join(
        candidates
    )

    values = []

    # 3 veya 6 taksit
    for match in re.finditer(
        (
            r"\b(\d{1,2})"
            r"\s*(?:ve|veya)\s*"
            r"(\d{1,2})"
            r"\s*taksit"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.extend(
            [
                match.group(1),
                match.group(2),
            ]
        )

    # 3-6-9-12 taksit
    for match in re.finditer(
        (
            r"("
            r"(?:\d{1,2}\s*[-,/]\s*)+"
            r"\d{1,2}"
            r")"
            r"\s*taksit"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.extend(
            re.findall(
                r"\d{1,2}",
                match.group(1)
            )
        )

    # 3 taksit / 12 aya varan taksit
    for match in re.finditer(
        (
            r"\b(\d{1,2})"
            r"\s*"
            r"(?:aya?\s+varan\s+)?"
            r"taksit\w*"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.append(
            match.group(1)
        )

    # Taksit sayıları: 3-6-9-12
    for match in re.finditer(
        (
            r"taksit\s+sayılar[ıi]"
            r"[^0-9]{0,20}"
            r"("
            r"(?:\d{1,2}\s*[-,/]\s*)+"
            r"\d{1,2}"
            r")"
        ),
        text,
        flags=re.IGNORECASE
    ):

        values.extend(
            re.findall(
                r"\d{1,2}",
                match.group(1)
            )
        )

    numbers = set()

    for value in values:

        if not value.isdigit():
            continue

        number = int(
            value
        )

        if 1 <= number <= 36:
            numbers.add(
                number
            )

    return [
        str(number)
        for number
        in sorted(numbers)
    ]


# ============================================================
# RATE
# ============================================================

def extract_rates(lines):

    kar_payi = []
    finansman = []

    for line in lines[:28]:

        financial_line = (
            normalize_financial_text(
                line
            )
        )

        normalized = normalize_match(
            financial_line
        )

        rates = re.findall(
            r"%([0-9]+(?:[.,][0-9]+)?)",
            financial_line
        )

        if not rates:
            continue

        for rate in rates:

            value = (
                "%"
                + rate.replace(
                    ".",
                    ","
                )
            )

            if any(
                marker in normalized
                for marker in (
                    "kar oran",
                    "kâr oran",
                    "kar pay",
                    "kâr pay",
                )
            ):

                kar_payi.append(
                    value
                )

            elif any(
                marker in normalized
                for marker in (
                    "vade fark",
                    "finansman oran",
                )
            ):

                finansman.append(
                    value
                )

    return (
        unique(kar_payi),
        unique(finansman),
    )


# ============================================================
# VADE
# ============================================================

def extract_maturity(
    category,
    title,
    lines
):

    if (
        category
        != "Finansman Kampanyaları"
    ):
        return []

    candidates = (
        [title]
        + extract_info_section(lines)
        + lines[:14]
    )

    result = []

    for line in candidates:

        normalized = normalize_match(
            line
        )

        # "vade farksız" bir oran/avantaj
        # ifadesidir, tek başına vade değil.
        if any(
            marker in normalized
            for marker in (
                "vade farksız",
                "vade farklı",
                "vade kısıtlam",
            )
        ):
            continue

        for match in re.finditer(
            r"\b(\d{1,2})\s*aya\s+varan",
            line,
            flags=re.IGNORECASE
        ):

            result.append(
                (
                    f"{match.group(1)} "
                    "aya varan"
                )
            )

        for match in re.finditer(
            r"\b(\d{1,2})\s*ay\s+vade",
            line,
            flags=re.IGNORECASE
        ):

            result.append(
                f"{match.group(1)} ay"
            )

    return unique(
        result
    )


# ============================================================
# MASRAF
# ============================================================

def extract_expenses(lines):

    result = []

    markers = (
        "komisyon",
        "tahsis ücreti",
        "ücret alın",
        "ücret tahsil",
        "masraf alın",
        "masraf tahsil",
        "masrafsız",
    )

    for line in lines:

        normalized = normalize_match(
            line
        )

        if any(
            marker in normalized
            for marker in markers
        ):

            result.append(
                normalize_financial_text(
                    line
                )
            )

    return unique(
        result
    )


# ============================================================
# TARGET
# ============================================================

def extract_targets(
    raw_text,
    lines
):

    result = []

    # ----------------------------------------------
    # Varsa sitedeki açık hedef-kitle bölümü
    # ----------------------------------------------

    target_section = (
        extract_target_section(
            lines
        )
    )

    for line in target_section:

        normalized = normalize_match(
            line
        )

        if len(line) > 300:
            continue

        if any(
            marker in normalized
            for marker in (
                "müşteri",
                "üye",
                "kart",
                "kitle",
            )
        ):

            result.append(
                line
            )

    normalized = normalize_match(
        raw_text
    )

    # ----------------------------------------------
    # HADI GOLD
    # ----------------------------------------------

    if (
        "hadi gold"
        in normalized
        and
        (
            "üye"
            in normalized
            or
            "üyeleri"
            in normalized
        )
    ):

        result.append(
            "Hadi Gold üyeleri"
        )

    # ----------------------------------------------
    # HADI BLACK
    # ----------------------------------------------

    black_patterns = [
        (
            r"yalnızca\s+hadi\s+black\s+"
            r"kredi\s+kart"
        ),
        (
            r"kampanya\s+hadi\s+black\s+"
            r"kredi\s+kart"
        ),
        (
            r"hadi\s+black\s+kredi\s+"
            r"kartı\s+ile\s+yapılacak"
        ),
        (
            r"hadi\s+black\s+kredi\s+"
            r"kartı\s+ile\s+gerçekleştir"
        ),
    ]

    if any(
        re.search(
            pattern,
            normalized
        )
        for pattern
        in black_patterns
    ):

        result.append(
            "Hadi Black Kredi Kartı sahipleri"
        )

    # ----------------------------------------------
    # GENEL HADI KREDI KARTI
    # ----------------------------------------------

    if (
        "hadi kredi kartları ile"
        in normalized
        or
        "hadi kredi kartı ile yapıl"
        in normalized
    ):

        result.append(
            "Hadi Kredi Kartı sahipleri"
        )

    # ----------------------------------------------
    # ÇOK KAZANANLAR
    # ----------------------------------------------

    if (
        "çok kazananlar kulüb"
        in normalized
        and
        (
            "üye"
            in normalized
            or
            "üyelik"
            in normalized
        )
    ):

        result.append(
            "Çok Kazananlar Kulübü üyeleri"
        )

    # ----------------------------------------------
    # TAKSİTLİ KREDİ LİMİTİ
    # ----------------------------------------------

    if re.search(
        (
            r"hadi\s+taksitli\s+"
            r"(?:alışveriş\s+)?"
            r"kredi\s+limiti"
            r"[^.]{0,180}"
            r"yeterli"
        ),
        normalized
    ):

        result.append(
            (
                "Hadi Taksitli Kredi limiti "
                "yeterli olan müşteriler"
            )
        )

    # ----------------------------------------------
    # YENİ MÜŞTERİ
    # ----------------------------------------------

    if (
        "ilk kez tom bank"
        in normalized
        or
        "ilk kez hadi müşter"
        in normalized
        or
        "daha önceden hiç tom bank müşter"
        in normalized
    ):

        result.append(
            (
                "İlk kez TOM Bank/Hadi "
                "müşterisi olacak kişiler"
            )
        )

    # ----------------------------------------------
    # SEÇİLİ KİTLE
    # ----------------------------------------------

    if (
        "kişiye özel"
        in normalized
        or
        "seçili kitle"
        in normalized
        or
        "seçili müşteri"
        in normalized
    ):

        result.append(
            "Seçili müşteri kitlesi"
        )

    # ----------------------------------------------
    # ÖZEL BANKACILIK
    # ----------------------------------------------

    if (
        "özel bankacılık"
        in normalized
        and
        (
            "classic"
            in normalized
            or
            "elite"
            in normalized
            or
            "prestige"
            in normalized
        )
    ):

        result.append(
            (
                "TOM Bank Hadi Özel Bankacılık "
                "müşterileri"
            )
        )

    return unique(
        result
    )[:10]


# ============================================================
# CURRENCY
# ============================================================

def extract_currency(lines):

    text = "\n".join(
        lines[:30]
    )

    if re.search(
        (
            r"(?:"
            r"\bTL\b"
            r"|₺"
            r"|Türk\s+Liras"
            r")"
        ),
        text,
        flags=re.IGNORECASE
    ):

        return ["TL"]

    return []


# ============================================================
# CONDITIONS
# ============================================================

def extract_conditions(
    lines,
    title
):

    keywords = (
        "geçerli",
        "gerekm",
        "zorunlu",
        "dahil",
        "hariç",
        "en fazla",
        "en az",
        "minimum",
        "maksimum",
        "sadece",
        "yalnızca",
        "faydalan",
        "kullan",
        "seçil",
        "iade",
        "iptal",
        "sınırlı",
        "müşteri",
        "ödeme",
        "kod",
        "işlem",
        "alışveriş",
        "kredi kartı",
        "kredi",
        "veresiye",
        "vade",
        "limit",
        "katılım",
    )

    title_n = normalize_match(
        title
    )

    result = []

    for line in lines:

        normalized = (
            normalize_match(line)
            .rstrip(":")
        )

        if normalize_match(
            line
        ) == title_n:
            continue

        if normalized in SECTION_HEADINGS:
            continue

        if (
            len(line) < 45
            and line.endswith(":")
        ):
            continue

        if any(
            keyword in normalized
            for keyword in keywords
        ):

            result.append(
                normalize_financial_text(
                    line
                )
            )

    return unique(
        result
    )


# ============================================================
# RECORD
# ============================================================

def extract_record(raw_record):

    title = normalize_text(
        raw_record.get(
            "kampanya_adi",
            ""
        )
    )

    url = normalize_text(
        raw_record.get(
            "kaynak_url",
            ""
        )
    )

    raw_text_original = (
        raw_record.get(
            "ham_metin",
            ""
        )
    )

    raw_text = normalize_text(
        raw_text_original
    )

    lines = content_lines(
        raw_text_original,
        title
    )

    (
        category,
        campaign_type,
    ) = classify_campaign(
        title,
        lines
    )

    (
        kar_payi_orani,
        finansman_orani,
    ) = extract_rates(
        lines
    )

    campaign_end = normalize_text(
        raw_record.get(
            "bitis_tarihi_iso",
            ""
        )
    )

    if not campaign_end:

        campaign_end = normalize_text(
            raw_record.get(
                "liste_bitis_tarihi",
                ""
            )
        )

    return {

        "banka":
            BANK_NAME,

        "kayit_turu":
            "kampanya",

        "urun_adi":
            title,

        "urun_kategorisi":
            category,

        "kar_payi_orani":
            kar_payi_orani,

        "finansman_orani":
            finansman_orani,

        # Harcama eşiği finansman tutarı değildir.
        "finansman_tutari":
            [],

        "vade":
            extract_maturity(
                category,
                title,
                lines
            ),

        "taksit_sayisi":
            extract_installments(
                title,
                lines
            ),

        "masraf_bilgisi":
            extract_expenses(
                lines
            ),

        "kampanya_turu":
            campaign_type,

        "kampanya_avantaji":
            extract_benefits(
                title,
                lines
            ),

        "kampanya_suresi":
            campaign_end,

        "hedef_kitle":
            extract_targets(
                raw_text,
                lines
            ),

        "para_birimi":
            extract_currency(
                lines
            ),

        "kosullar":
            extract_conditions(
                lines,
                title
            ),

        "kaynak_url":
            url,

        "ham_metin":
            raw_text,
    }


# ============================================================
# SCHEMA VALIDATION
# ============================================================

def validate_schema(
    record,
    index
):

    errors = []

    if (
        list(record.keys())
        != SCHEMA_KEYS
    ):

        errors.append(
            (
                f"[{index}] "
                "schema key/order uyuşmuyor."
            )
        )

    for field in LIST_FIELDS:

        if not isinstance(
            record.get(field),
            list
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} list değil."
                )
            )

    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(field),
            str
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} string değil."
                )
            )

    if (
        record.get(
            "kayit_turu"
        )
        != "kampanya"
    ):

        errors.append(
            (
                f"[{index}] "
                "kayit_turu yanlış."
            )
        )

    if "TRY" in json.dumps(
        record,
        ensure_ascii=False
    ):

        errors.append(
            f"[{index}] TRY bulundu."
        )

    if any(
        currency != "TL"
        for currency
        in record.get(
            "para_birimi",
            []
        )
    ):

        errors.append(
            (
                f"[{index}] "
                "desteklenmeyen para birimi."
            )
        )

    if (
        not record.get(
            "urun_adi"
        )
        or
        not record.get(
            "kaynak_url"
        )
        or
        not record.get(
            "ham_metin"
        )
    ):

        errors.append(
            (
                f"[{index}] "
                "provenance alanı boş."
            )
        )

    return errors


# ============================================================
# SEMANTIC VALIDATION
# ============================================================

def contains(values, text):

    search = normalize_match(
        text
    )

    return any(
        search
        in normalize_match(
            item
        )
        for item in values
    )


def semantic_validation(records):

    errors = []

    by_slug = {
        get_slug(
            record[
                "kaynak_url"
            ]
        ):
        record
        for record
        in records
    }

    # ========================================================
    # GLOBAL:
    # Negatif cümle avantaj olamaz
    # ========================================================

    for record in records:

        for benefit in record[
            "kampanya_avantaji"
        ]:

            if is_negative_benefit(
                benefit
            ):

                errors.append(
                    (
                        get_slug(
                            record[
                                "kaynak_url"
                            ]
                        )
                        + " -> negatif/operasyonel "
                        + "cümle avantaj alanında: "
                        + benefit
                    )
                )

    # ========================================================
    # A101 %3
    # ========================================================

    slug = (
        "a101de-her-alisveriste-"
        "3-nakit-iade"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if (
            item[
                "urun_kategorisi"
            ]
            != "Kart Kampanyaları"
        ):

            errors.append(
                (
                    "A101 %3 -> kategori yanlış: "
                    f"{item['urun_kategorisi']}"
                )
            )

        if (
            item[
                "kampanya_turu"
            ]
            != "Kart Kampanyası"
        ):

            errors.append(
                (
                    "A101 %3 -> kampanya türü yanlış."
                )
            )

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "%3"
        ):

            errors.append(
                "A101 %3 -> avantaj bulunamadı."
            )

        if not contains(
            item[
                "hedef_kitle"
            ],
            "Hadi Gold"
        ):

            errors.append(
                (
                    "A101 %3 -> "
                    "Hadi Gold hedef kitle yok."
                )
            )

        if (
            item[
                "kampanya_suresi"
            ]
            != "2026-12-31"
        ):

            errors.append(
                (
                    "A101 %3 -> tarih yanlış: "
                    f"{item['kampanya_suresi']}"
                )
            )

    # ========================================================
    # TOM1500
    # ========================================================

    slug = (
        "toplam-1500-tl-hos-geldin-"
        "hediyesi-tom1500-koduyla-"
        "musterimiz-ol-1500-tl-"
        "senin-olsun"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if (
            item[
                "urun_kategorisi"
            ]
            != "Yeni Müşteri Kampanyaları"
        ):

            errors.append(
                "TOM1500 -> kategori yanlış."
            )

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "1.500 TL"
        ):

            errors.append(
                (
                    "TOM1500 -> "
                    "1.500 TL avantajı yok."
                )
            )

        if not any(
            "ilk kez"
            in normalize_match(target)
            for target
            in item[
                "hedef_kitle"
            ]
        ):

            errors.append(
                (
                    "TOM1500 -> "
                    "yeni müşteri hedef kitlesi yok."
                )
            )

    # ========================================================
    # KLIMA 12 TAKSIT
    # ========================================================

    slug = (
        "hadi-alisveris-kredisi-ile-"
        "klima-supurge-ve-"
        "televizyonlarda-vade-"
        "farksiz-12-taksit"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if (
            item[
                "urun_kategorisi"
            ]
            != "Finansman Kampanyaları"
        ):

            errors.append(
                "Klima -> kategori yanlış."
            )

        if "12" not in item[
            "taksit_sayisi"
        ]:

            errors.append(
                "Klima -> 12 taksit yok."
            )

        if item[
            "kar_payi_orani"
        ]:

            errors.append(
                (
                    "Klima -> vade farksız "
                    "ifadesinden kâr payı üretildi."
                )
            )

    # ========================================================
    # RESTODERM
    # ========================================================

    slug = (
        "hadi-black-karti-ile-"
        "restoderm-alisverislerinde-"
        "30-indirim"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if (
            item[
                "urun_kategorisi"
            ]
            != "Kart Kampanyaları"
        ):

            errors.append(
                "Restoderm -> kategori yanlış."
            )

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "%30"
        ):

            errors.append(
                "Restoderm -> %30 yok."
            )

    # ========================================================
    # SÜT %50
    # ========================================================

    slug = (
        "a101lerde-sut-urunleri-"
        "harcamalarinda-50-"
        "hediye-bakiye-kazan"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if not contains(
            item[
                "kampanya_avantaji"
            ],
            "%50"
        ):

            errors.append(
                "Süt -> %50 avantaj yok."
            )

    # ========================================================
    # MTV
    # ========================================================

    slug = "vergi-kampanyasi"

    if slug in by_slug:

        item = by_slug[slug]

        if item[
            "taksit_sayisi"
        ] != ["3"]:

            errors.append(
                (
                    "MTV -> taksit yanlış: "
                    f"{item['taksit_sayisi']}"
                )
            )

    # ========================================================
    # GİYİM
    # ========================================================

    slug = (
        "giyim-alisverislerinde-"
        "kampanya"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if item[
            "taksit_sayisi"
        ] != [
            "3",
            "6",
        ]:

            errors.append(
                (
                    "Giyim -> taksit yanlış: "
                    f"{item['taksit_sayisi']}"
                )
            )

    # ========================================================
    # ECZANE
    # ========================================================

    slug = (
        "eczanelerde-hadi-saglik-"
        "kredisi-ile-0-vade-farki-"
        "ile-6-taksit"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if (
            item[
                "urun_kategorisi"
            ]
            != "Finansman Kampanyaları"
        ):

            errors.append(
                "Eczane -> kategori yanlış."
            )

        if not contains(
            item[
                "finansman_orani"
            ],
            "%0"
        ):

            errors.append(
                (
                    "Eczane -> "
                    "explicit %0 vade farkı yok."
                )
            )

        if item[
            "kar_payi_orani"
        ]:

            errors.append(
                (
                    "Eczane -> %0 yanlışlıkla "
                    "kâr payına yazıldı."
                )
            )

    # ========================================================
    # SAMSUNG TELEFON
    # ========================================================

    slug = (
        "tom-bank-hadi-taksitli-"
        "alisveris-kredisi-ile-"
        "samsung-cep-telefonlarinda-"
        "3-taksit-firsati"
    )

    if slug in by_slug:

        item = by_slug[slug]

        if item[
            "taksit_sayisi"
        ] != ["3"]:

            errors.append(
                (
                    "Samsung telefon -> "
                    "yalnızca 3 taksit olmalı: "
                    f"{item['taksit_sayisi']}"
                )
            )

    return errors


# ============================================================
# LOAD
# ============================================================

print("=" * 110)
print("TOM KATILIM - CAMPAIGN EXTRACTOR V2")
print("=" * 110)

print("Input :", INPUT_FILE)
print("Output:", OUTPUT_FILE)
print()


with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as file:

    raw_data = json.load(file)


raw_records = raw_data.get(
    "kampanyalar",
    []
)

EXPECTED_COUNT = raw_data.get(
    "aktif_kampanya_sayisi",
    53
)


# ============================================================
# ACTIVE
# ============================================================

active_records = [
    record
    for record in raw_records
    if (
        normalize_match(
            record.get(
                "liste_durumu",
                ""
            )
        )
        == "aktif"
    )
]


# ============================================================
# EXTRACTION
# ============================================================

extracted = []

extraction_errors = []
schema_errors = []


for index, raw_record in enumerate(
    active_records,
    start=1
):

    try:

        record = extract_record(
            raw_record
        )

        extracted.append(
            record
        )

        schema_errors.extend(
            validate_schema(
                record,
                index
            )
        )

    except Exception as error:

        extraction_errors.append(
            (
                f"[{index}] "
                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ============================================================
# DUPLICATES
# ============================================================

urls = [
    record[
        "kaynak_url"
    ]
    for record
    in extracted
]

duplicate_url_count = (
    len(urls)
    -
    len(set(urls))
)


# ============================================================
# SEMANTIC
# ============================================================

semantic_errors = (
    semantic_validation(
        extracted
    )
)


if (
    len(active_records)
    != EXPECTED_COUNT
):

    semantic_errors.append(
        (
            "Aktif RAW sayısı yanlış: "
            f"{len(active_records)} "
            f"!= {EXPECTED_COUNT}"
        )
    )


if (
    len(extracted)
    != EXPECTED_COUNT
):

    semantic_errors.append(
        (
            "Extracted sayısı yanlış: "
            f"{len(extracted)} "
            f"!= {EXPECTED_COUNT}"
        )
    )


if duplicate_url_count:

    semantic_errors.append(
        (
            "Duplicate URL var: "
            f"{duplicate_url_count}"
        )
    )


# ============================================================
# SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        extracted,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# RESULT
# ============================================================

total_errors = (
    len(extraction_errors)
    +
    len(schema_errors)
    +
    len(semantic_errors)
)


print()
print("=" * 110)
print("CAMPAIGN EXTRACTION V2 SONUCU")
print("=" * 110)

print(
    "Beklenen aktif :",
    EXPECTED_COUNT
)

print(
    "Clean RAW       :",
    len(raw_records)
)

print(
    "Aktif RAW       :",
    len(active_records)
)

print(
    "Extracted       :",
    len(extracted)
)

print(
    "Duplicate URL   :",
    duplicate_url_count
)

print(
    "Extraction error:",
    len(extraction_errors)
)

print(
    "Schema error    :",
    len(schema_errors)
)

print(
    "Semantic error  :",
    len(semantic_errors)
)

print(
    "Toplam error    :",
    total_errors
)


# ============================================================
# ERRORS
# ============================================================

if extraction_errors:

    print()
    print("EXTRACTION HATALARI:")

    for error in extraction_errors:
        print("-", error)


if schema_errors:

    print()
    print("SCHEMA HATALARI:")

    for error in schema_errors:
        print("-", error)


if semantic_errors:

    print()
    print("SEMANTİK HATALAR:")

    for error in semantic_errors:
        print("-", error)


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

print()
print("=" * 110)
print("KATEGORİ DAĞILIMI")
print("=" * 110)


categories = {}


for record in extracted:

    category = record[
        "urun_kategorisi"
    ]

    categories[
        category
    ] = (
        categories.get(
            category,
            0
        )
        + 1
    )


for category, count in sorted(
    categories.items()
):

    print(
        f"{category}: {count}"
    )


# ============================================================
# FIRST RECORD
# ============================================================

if extracted:

    print()
    print("=" * 110)
    print("İLK PROCESSED KAYIT")
    print("=" * 110)

    print(
        json.dumps(
            extracted[0],
            ensure_ascii=False,
            indent=2
        )
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 110)

if total_errors == 0:

    print(
        "SONUÇ: TOM KATILIM "
        "CAMPAIGN EXTRACTION V2 "
        "BAŞARILI ✅"
    )

    print(
        (
            f"{len(extracted)} aktif kampanya "
            "18-key final schema'ya "
            "başarıyla dönüştürüldü ✅"
        )
    )

else:

    print(
        "SONUÇ: TOM KATILIM "
        "CAMPAIGN EXTRACTION V2 "
        "KONTROL GEREKİYOR ❌"
    )


print()
print(
    "JSON:",
    OUTPUT_FILE
)

print("=" * 110)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    OUTPUT_FILE
)

TOM KATILIM - CAMPAIGN EXTRACTOR V2
Input : /content/tom_katilim_kampanyalar_clean.json
Output: /content/tom_katilim_kampanya_extracted_v2.json


CAMPAIGN EXTRACTION V2 SONUCU
Beklenen aktif : 53
Clean RAW       : 81
Aktif RAW       : 53
Extracted       : 53
Duplicate URL   : 0
Extraction error: 0
Schema error    : 0
Semantic error  : 0
Toplam error    : 0

KATEGORİ DAĞILIMI
Alışveriş Kampanyaları: 10
Finansman Kampanyaları: 15
Kart Kampanyaları: 26
Sigorta Kampanyaları: 1
Yeni Müşteri Kampanyaları: 1

İLK PROCESSED KAYIT
{
  "banka": "T.O.M. Katılım Bankası",
  "kayit_turu": "kampanya",
  "urun_adi": "A101'de her alışverişte %3'e varan nakit iade!",
  "urun_kategorisi": "Kart Kampanyaları",
  "kar_payi_orani": [],
  "finansman_orani": [],
  "finansman_tutari": [],
  "vade": [],
  "taksit_sayisi": [],
  "masraf_bilgisi": [],
  "kampanya_turu": "Kart Kampanyası",
  "kampanya_avantaji": [
    "A101'de her alışverişte %3'e varan nakit iade!",
    "Kampanya kapsamında bir takvim ayında en 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

FILE = "/content/tom_katilim_kampanya_extracted_v2.json"

with open(FILE, "r", encoding="utf-8") as f:
    data = json.load(f)


wanted = [
    "toplam-1500-tl-hos-geldin-hediyesi",
    "klima-supurge-ve-televizyonlarda",
    "vergi-kampanyasi",
    "giyim-alisverislerinde-kampanya",
    "eczanelerde-hadi-saglik-kredisi",
    "samsung-cep-telefonlarinda-3-taksit",
]


for record in data:

    url = record["kaynak_url"]

    if any(x in url for x in wanted):

        print("=" * 120)
        print(record["urun_adi"])
        print("=" * 120)

        print(
            "Kategori       :",
            record["urun_kategorisi"]
        )

        print(
            "Tür            :",
            record["kampanya_turu"]
        )

        print(
            "Kâr Payı       :",
            record["kar_payi_orani"]
        )

        print(
            "Finansman Oranı:",
            record["finansman_orani"]
        )

        print(
            "Vade           :",
            record["vade"]
        )

        print(
            "Taksit         :",
            record["taksit_sayisi"]
        )

        print(
            "Avantaj        :",
            record["kampanya_avantaji"]
        )

        print(
            "Hedef Kitle    :",
            record["hedef_kitle"]
        )

        print(
            "Süre           :",
            record["kampanya_suresi"]
        )

        print()

Toplam 1500 TL hoş geldin hediyesi! TOM1500 koduyla müşterimiz ol, 1500 TL senin olsun!
Kategori       : Yeni Müşteri Kampanyaları
Tür            : Yeni Müşteri Kampanyası
Kâr Payı       : []
Finansman Oranı: []
Vade           : []
Taksit         : []
Avantaj        : ['Toplam 1500 TL hoş geldin hediyesi! TOM1500 koduyla müşterimiz ol, 1500 TL senin olsun!', '20.8.2026 - 31.8.2026 tarihleri arasında, TOM1500 kodu ile banka müşterisi ol. Hadi Black Kredi Kartın ile gerçekleştireceğin 1.000 TL ve üzeri harcamalardan toplam 1.500 TL Hediye Bakiye kazan!', 'Kazanılan Hediye Bakiye, ilgili harcamanın kampanya koşullarını sağlaması halinde, kampanya bitiş tarihinden itibaren 5 iş günü içerisinde hediye bakiye hesabına yüklenecektir.', "Yüklenmiş olan Hediye Bakiyeler yalnızca A101'lerde kullanılabilecek bir harcama bakiyesi olup, nakit çekim yapılamaz."]
Hedef Kitle    : ['“TOM1500” davet koduyla, ilk kez TOM Bank müşterisi olan müşteriler kampanyadan yararlanabilir.', 'Hadi Black Kredi Kart

In [ ]:
import json
import re
from urllib.parse import urlparse

from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

CLEAN_FILE = (
    "/content/"
    "tom_katilim_kampanyalar_clean.json"
)

V2_FILE = (
    "/content/"
    "tom_katilim_kampanya_extracted_v2.json"
)

OUTPUT_FILE = (
    "/content/"
    "tom_katilim_kampanya_extracted_v3.json"
)


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(value):

    value = str(value or "")

    replacements = {
        "’": "'",
        "‘": "'",
        "–": "-",
        "—": "-",
        "\xa0": " ",
    }

    for old, new in replacements.items():
        value = value.replace(old, new)

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    return value.strip()


def normalize_match(value):

    value = normalize_text(value)

    value = value.replace(
        "İ",
        "i"
    )

    value = value.replace(
        "I",
        "ı"
    )

    value = value.casefold()

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize_financial(value):

    value = normalize_text(value)

    # 7.5% -> %7.5
    value = re.sub(
        (
            r"(?<![%\d])"
            r"(\d+(?:[.,]\d+)?)"
            r"\s*%"
        ),
        r"%\1",
        value
    )

    # % 7.5 -> %7.5
    value = re.sub(
        r"%\s+(\d+(?:[.,]\d+)?)",
        r"%\1",
        value
    )

    # %7.5 -> %7,5
    value = re.sub(
        r"%(\d+)\.(\d+)",
        r"%\1,\2",
        value
    )

    return normalize_text(
        value
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = normalize_text(
            value
        )

        if not value:
            continue

        key = normalize_match(
            value
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        result.append(
            value
        )

    return result


def get_slug(url):

    path = urlparse(
        str(url or "")
    ).path.rstrip("/")

    if not path:
        return ""

    return path.split("/")[-1]


# ============================================================
# RAW LINES
# ============================================================

def get_lines(raw_text):

    lines = []

    for raw_line in str(
        raw_text or ""
    ).splitlines():

        line = normalize_text(
            raw_line
        )

        if not line:
            continue

        normalized = (
            normalize_match(line)
            .rstrip(":")
        )

        if normalized in {
            "hemen indir",
            "daha fazla göster",
        }:
            continue

        lines.append(
            line
        )

    return unique(
        lines
    )


# ============================================================
# KAMPANYA BİLGİLERİ
# ============================================================

SECTION_NAMES = {
    "kampanya tarihleri",
    "kampanya bilgileri",
    "kampanya detayları",
    "kampanya kuralları",
    "kampanya koşulları",
    "kampanya şartları",
    "kampanyadan yararlanabilecek kişiler",
    "kampanyadan yararlanacak kişiler",
}


def get_section(
    lines,
    section_name
):

    result = []
    active = False

    for line in lines:

        normalized = (
            normalize_match(line)
            .rstrip(":")
        )

        if normalized == section_name:

            active = True
            continue

        if (
            active
            and normalized in SECTION_NAMES
        ):
            break

        if active:
            result.append(
                line
            )

    return result


# ============================================================
# INTRO
#
# Kampanya Detayları başlamadan önceki
# kampanyaya özgü açıklamalar.
# ============================================================

def get_intro_lines(
    lines,
    title
):

    result = []

    title_n = normalize_match(
        title
    )

    title_seen = False

    for line in lines:

        normalized = (
            normalize_match(line)
            .rstrip(":")
        )

        if normalized == title_n:

            if not title_seen:
                title_seen = True
                continue

        if normalized in {
            "kampanya detayları",
            "kampanya kuralları",
            "kampanya koşulları",
            "kampanya şartları",
        }:
            break

        if normalized in SECTION_NAMES:
            continue

        result.append(
            line
        )

    return unique(
        result
    )


# ============================================================
# CATEGORY V3
# ============================================================

def classify_v3(
    title,
    raw_text,
    lines
):

    title_n = normalize_match(
        title
    )

    raw_n = normalize_match(
        raw_text
    )

    info_n = normalize_match(
        " ".join(
            get_section(
                lines,
                "kampanya bilgileri"
            )
        )
    )

    intro_n = normalize_match(
        " ".join(
            get_intro_lines(
                lines,
                title
            )
        )
    )

    # ========================================================
    # 1. ÖZEL BANKACILIK
    # ========================================================

    private_patterns = [
        (
            "özel bankacılık hadi birikim "
            "segmentine göre"
        ),
        (
            "kampanya özel bankacılık "
            "classic"
        ),
        (
            "özel bankacılık classic, "
            "elite"
        ),
    ]

    if any(
        pattern in raw_n
        for pattern
        in private_patterns
    ):

        return (
            "Özel Bankacılık Kampanyaları",
            "Özel Bankacılık Kampanyası",
        )

    # ========================================================
    # 2. YENİ MÜŞTERİ
    # ========================================================

    primary = (
        title_n
        + " "
        + info_n
        + " "
        + intro_n
    )

    if any(
        marker in primary
        for marker in [
            "müşterimiz ol",
            "ilk kez tom bank",
            "ilk kez hadi müşter",
            "hoş geldin",
            "davet kodu",
        ]
    ):

        return (
            "Yeni Müşteri Kampanyaları",
            "Yeni Müşteri Kampanyası",
        )

    # ========================================================
    # 3. SİGORTA
    # ========================================================

    if "sigorta" in title_n:

        return (
            "Sigorta Kampanyaları",
            "Sigorta Kampanyası",
        )

    # ========================================================
    # 4. FİNANSMAN
    #
    # Negatif cümleleri çıkartıyoruz.
    # ========================================================

    finance_lines = []

    negative_finance_markers = (
        "kazanımı yok",
        "kazanım yok",
        "geçerli değildir",
        "dahil değildir",
        "kullanılamaz",
    )

    for line in lines:

        normalized = normalize_match(
            line
        )

        if any(
            marker in normalized
            for marker
            in negative_finance_markers
        ):
            continue

        finance_lines.append(
            line
        )

    finance_text = normalize_match(
        " ".join(
            finance_lines[:20]
        )
    )

    finance_primary = (
        title_n
        + " "
        + info_n
        + " "
        + finance_text
    )

    finance_markers = [
        "taksitli alışveriş kredisi",
        "mağazadan alışveriş kredisi",
        "alışveriş kredisi",
        "taksitli sağlık kredisi",
        "sağlık kredisi",
        "hadi taksitli kredi",
        "hadi veresiye",
        "veresiye ile",
    ]

    if any(
        marker in finance_primary
        for marker
        in finance_markers
    ):

        return (
            "Finansman Kampanyaları",
            "Finansman Kampanyası",
        )

    # ========================================================
    # 5. KART
    # ========================================================

    card_text = (
        title_n
        + " "
        + info_n
        + " "
        + intro_n
    )

    if any(
        marker in card_text
        for marker in [
            "kredi kartı",
            "black kart",
            "hadi kart",
            "mastercard",
            "hesap kartı",
        ]
    ):

        return (
            "Kart Kampanyaları",
            "Kart Kampanyası",
        )

    # ========================================================
    # 6. ALIŞVERİŞ
    # ========================================================

    return (
        "Alışveriş Kampanyaları",
        "Alışveriş Kampanyası",
    )


# ============================================================
# RATE V3
# ============================================================

def extract_rates_v3(
    lines
):

    kar_payi = []
    finansman = []

    for line in lines:

        financial_line = (
            normalize_financial(
                line
            )
        )

        normalized = normalize_match(
            financial_line
        )

        rates = re.findall(
            (
                r"%"
                r"(\d+(?:[.,]\d+)?)"
            ),
            financial_line
        )

        if not rates:
            continue

        for rate in rates:

            rate = rate.replace(
                ".",
                ","
            )

            value = "%" + rate

            # ----------------------------------------------
            # AÇIK KÂR PAYI
            # ----------------------------------------------

            if any(
                marker in normalized
                for marker in (
                    "kâr payı oranı",
                    "kar payı oranı",
                    "kâr oranı",
                    "kar oranı",
                )
            ):

                kar_payi.append(
                    value
                )

            # ----------------------------------------------
            # AÇIK VADE FARKI
            # ----------------------------------------------

            if any(
                marker in normalized
                for marker in (
                    "vade farkı",
                    "vade farkıyla",
                    "vade farkiyla",
                    "finansman oranı",
                )
            ):

                finansman.append(
                    value
                )

    return (
        unique(
            kar_payi
        ),
        unique(
            finansman
        ),
    )


# ============================================================
# BENEFIT V3
# ============================================================

OPERATIONAL_MARKERS = (
    "kazanılan ",
    "yüklen",
    "yansıt",
    "geri alın",
    "nakit çek",
    "kullanım süresi",
    "kullanılmayan",
    "kullanılabilecek",
    "kazanımı yok",
    "kazanım yok",
    "kazanılmaz",
    "kazanılamaz",
    "iade verilmeyecek",
    "iade verilmeyecektir",
    "iptal edil",
    "iptal/iade",
    "iptal veya iade",
    "faydalanmak için",
    "faydalanabilmek için",
    "gerekmektedir",
    "gereklidir",
    "yalnızca",
    "sadece",
    "saklı tutar",
    "detaylı bilgi",
    "tıkla",
    "başvuru için",
    "geçerlidir",
)


def is_operational(
    line
):

    normalized = normalize_match(
        line
    )

    return any(
        marker in normalized
        for marker
        in OPERATIONAL_MARKERS
    )


def has_real_benefit(
    line
):

    normalized = normalize_match(
        line
    )

    positive_patterns = (
        "indirim",
        "nakit iade",
        "hediye bakiye",
        "hediye",
        "bedava",
        "vade farksız",
        "peşin fiyatına",
        "taksit fırsat",
        "taksitte öde",
        "taksitte ode",
        "iade kazan",
        "kazanabil",
        "kazanman",
        "kat kazan",
    )

    if any(
        marker in normalized
        for marker
        in positive_patterns
    ):
        return True

    # %... + avantaj tipi
    if (
        "%"
        in line
        and
        any(
            marker in normalized
            for marker
            in (
                "iade",
                "indirim",
                "vade fark",
            )
        )
    ):
        return True

    return False


def extract_benefits_v3(
    title,
    lines
):

    result = []

    # --------------------------------------------------------
    # Başlık kampanya avantajını açıkça söylüyorsa ekle
    # --------------------------------------------------------

    if (
        has_real_benefit(
            title
        )
        and
        not is_operational(
            title
        )
    ):

        result.append(
            normalize_financial(
                title
            )
        )

    # --------------------------------------------------------
    # Kampanya Bilgileri en güvenilir bölüm
    # --------------------------------------------------------

    info_lines = get_section(
        lines,
        "kampanya bilgileri"
    )

    for line in info_lines:

        if is_operational(
            line
        ):
            continue

        if has_real_benefit(
            line
        ):

            result.append(
                normalize_financial(
                    line
                )
            )

    # --------------------------------------------------------
    # Kampanya Bilgileri yoksa intro kısmı
    # --------------------------------------------------------

    if not info_lines:

        intro_lines = get_intro_lines(
            lines,
            title
        )

        for line in intro_lines[:12]:

            if is_operational(
                line
            ):
                continue

            if has_real_benefit(
                line
            ):

                result.append(
                    normalize_financial(
                        line
                    )
                )

    # --------------------------------------------------------
    # Detail içinden yalnızca çok net kazanım satırları
    # --------------------------------------------------------

    detail_lines = get_section(
        lines,
        "kampanya detayları"
    )

    for line in detail_lines:

        if is_operational(
            line
        ):
            continue

        normalized = normalize_match(
            line
        )

        # "en fazla 125 TL iade kazanılabilir"
        # gibi gerçekten faydayı tanımlayan satırlar.
        strong = (
            "iade kazanılabilir"
            in normalized

            or

            "iade kazanabilir"
            in normalized

            or

            "hediye bakiye kazan"
            in normalized

            or

            "vade farksız"
            in normalized

            or

            "peşin fiyatına"
            in normalized
        )

        if strong:

            result.append(
                normalize_financial(
                    line
                )
            )

        if len(result) >= 8:
            break

    return unique(
        result
    )[:8]


# ============================================================
# LOAD
# ============================================================

with open(
    CLEAN_FILE,
    "r",
    encoding="utf-8"
) as f:

    clean_data = json.load(
        f
    )


with open(
    V2_FILE,
    "r",
    encoding="utf-8"
) as f:

    records = json.load(
        f
    )


raw_by_slug = {}

for raw in clean_data[
    "kampanyalar"
]:

    if (
        normalize_match(
            raw.get(
                "liste_durumu",
                ""
            )
        )
        != "aktif"
    ):
        continue

    slug = get_slug(
        raw.get(
            "kaynak_url",
            ""
        )
    )

    raw_by_slug[
        slug
    ] = raw


# ============================================================
# V3 UPDATE
# ============================================================

for record in records:

    slug = get_slug(
        record[
            "kaynak_url"
        ]
    )

    raw = raw_by_slug.get(
        slug
    )

    if not raw:
        continue

    raw_text = raw.get(
        "ham_metin",
        ""
    )

    lines = get_lines(
        raw_text
    )

    # --------------------------------------------------------
    # CATEGORY
    # --------------------------------------------------------

    (
        category,
        campaign_type,
    ) = classify_v3(
        record[
            "urun_adi"
        ],
        raw_text,
        lines
    )

    record[
        "urun_kategorisi"
    ] = category

    record[
        "kampanya_turu"
    ] = campaign_type

    # --------------------------------------------------------
    # RATES
    # --------------------------------------------------------

    (
        kar_payi,
        finansman,
    ) = extract_rates_v3(
        lines
    )

    record[
        "kar_payi_orani"
    ] = kar_payi

    record[
        "finansman_orani"
    ] = finansman

    # --------------------------------------------------------
    # BENEFITS
    # --------------------------------------------------------

    record[
        "kampanya_avantaji"
    ] = extract_benefits_v3(
        record[
            "urun_adi"
        ],
        lines
    )


# ============================================================
# VALIDATION
# ============================================================

errors = []


if len(records) != 53:

    errors.append(
        (
            "Kayıt sayısı "
            f"53 değil: {len(records)}"
        )
    )


urls = [
    x[
        "kaynak_url"
    ]
    for x in records
]


if len(urls) != len(
    set(urls)
):

    errors.append(
        "Duplicate URL bulundu."
    )


by_slug = {
    get_slug(
        x[
            "kaynak_url"
        ]
    ):
    x
    for x in records
}


# ============================================================
# SAMSUNG %0 KÂR PAYI
# ============================================================

samsung_slug = (
    "tom-bank-hadi-taksitli-"
    "alisveris-kredisi-ile-"
    "samsung-cep-telefonlarinda-"
    "3-taksit-firsati"
)


if samsung_slug in by_slug:

    samsung = by_slug[
        samsung_slug
    ]

    if samsung[
        "kar_payi_orani"
    ] != ["%0"]:

        errors.append(
            (
                "Samsung -> "
                "kâr payı %0 bulunamadı: "
                f"{samsung['kar_payi_orani']}"
            )
        )

    if samsung[
        "taksit_sayisi"
    ] != ["3"]:

        errors.append(
            (
                "Samsung -> "
                "taksit yanlış: "
                f"{samsung['taksit_sayisi']}"
            )
        )


# ============================================================
# ECZANE
# ============================================================

eczane_slug = (
    "eczanelerde-hadi-saglik-"
    "kredisi-ile-0-vade-farki-"
    "ile-6-taksit"
)


if eczane_slug in by_slug:

    eczane = by_slug[
        eczane_slug
    ]

    if "%0" not in eczane[
        "finansman_orani"
    ]:

        errors.append(
            (
                "Eczane -> "
                "%0 finansman oranı yok."
            )
        )

    if eczane[
        "taksit_sayisi"
    ] != [
        "3",
        "6",
    ]:

        errors.append(
            (
                "Eczane -> "
                "taksit yanlış: "
                f"{eczane['taksit_sayisi']}"
            )
        )


# ============================================================
# ÖZEL BANKACILIK
# ============================================================

private_names = [
    (
        "restoran-harcamalarinda-"
        "her-ay-10-000-tlye-varan-iade"
    ),
    (
        "ucak-bileti-alimlarinda-"
        "ayda-3-500-tl-kazan"
    ),
]


for slug in private_names:

    if slug not in by_slug:

        errors.append(
            (
                "Özel Bankacılık "
                f"kaydı bulunamadı: {slug}"
            )
        )

        continue

    item = by_slug[
        slug
    ]

    if (
        item[
            "urun_kategorisi"
        ]
        != "Özel Bankacılık Kampanyaları"
    ):

        errors.append(
            (
                f"{slug} -> "
                "Özel Bankacılık değil: "
                f"{item['urun_kategorisi']}"
            )
        )


# ============================================================
# A101
# ============================================================

a101_slug = (
    "a101de-her-alisveriste-"
    "3-nakit-iade"
)


if a101_slug in by_slug:

    a101 = by_slug[
        a101_slug
    ]

    if (
        a101[
            "urun_kategorisi"
        ]
        != "Kart Kampanyaları"
    ):

        errors.append(
            (
                "A101 -> kategori yanlış: "
                f"{a101['urun_kategorisi']}"
            )
        )


# ============================================================
# NEGATIVE BENEFIT GLOBAL
# ============================================================

for record in records:

    for benefit in record[
        "kampanya_avantaji"
    ]:

        if is_operational(
            benefit
        ):

            errors.append(
                (
                    get_slug(
                        record[
                            "kaynak_url"
                        ]
                    )
                    +
                    " -> operasyonel cümle "
                    "avantaj alanında: "
                    + benefit
                )
            )


# ============================================================
# SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# CATEGORY COUNTS
# ============================================================

categories = {}

for record in records:

    category = record[
        "urun_kategorisi"
    ]

    categories[
        category
    ] = (
        categories.get(
            category,
            0
        )
        + 1
    )


# ============================================================
# RESULT
# ============================================================

print(
    "=" * 110
)

print(
    "TOM KATILIM - "
    "CAMPAIGN FINALIZER V3"
)

print(
    "=" * 110
)

print(
    "Kayıt           :",
    len(records)
)

print(
    "Duplicate URL   :",
    len(urls)
    - len(set(urls))
)

print(
    "Validation error:",
    len(errors)
)

print()


print(
    "=" * 110
)

print(
    "KATEGORİ DAĞILIMI"
)

print(
    "=" * 110
)


for category, count in sorted(
    categories.items()
):

    print(
        f"{category}: {count}"
    )


# ============================================================
# CRITICAL RECORDS
# ============================================================

print()
print(
    "=" * 110
)

print(
    "KRİTİK KONTROLLER"
)

print(
    "=" * 110
)


critical_slugs = [
    a101_slug,
    samsung_slug,
    eczane_slug,
    *private_names,
]


for slug in critical_slugs:

    if slug not in by_slug:
        continue

    item = by_slug[
        slug
    ]

    print()
    print(
        item[
            "urun_adi"
        ]
    )

    print(
        "Kategori       :",
        item[
            "urun_kategorisi"
        ]
    )

    print(
        "Kâr Payı       :",
        item[
            "kar_payi_orani"
        ]
    )

    print(
        "Finansman Oranı:",
        item[
            "finansman_orani"
        ]
    )

    print(
        "Taksit         :",
        item[
            "taksit_sayisi"
        ]
    )

    print(
        "Avantaj        :",
        item[
            "kampanya_avantaji"
        ]
    )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print(
        "=" * 110
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 110
    )

    for error in errors:

        print(
            "-",
            error
        )


print()
print(
    "=" * 110
)

if not errors:

    print(
        "SONUÇ: TOM KATILIM "
        "CAMPAIGN V3 BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: TOM KATILIM "
        "CAMPAIGN V3 "
        "KONTROL GEREKİYOR ❌"
    )

print(
    "JSON:",
    OUTPUT_FILE
)

print(
    "=" * 110
)


files.download(
    OUTPUT_FILE
)

TOM KATILIM - CAMPAIGN FINALIZER V3
Kayıt           : 53
Duplicate URL   : 0
Validation error: 1

KATEGORİ DAĞILIMI
Alışveriş Kampanyaları: 2
Finansman Kampanyaları: 23
Kart Kampanyaları: 23
Sigorta Kampanyaları: 1
Yeni Müşteri Kampanyaları: 2
Özel Bankacılık Kampanyaları: 2

KRİTİK KONTROLLER

A101'de her alışverişte %3'e varan nakit iade!
Kategori       : Kart Kampanyaları
Kâr Payı       : []
Finansman Oranı: []
Taksit         : []
Avantaj        : ["A101'de her alışverişte %3'e varan nakit iade!", 'Kampanya kapsamında bir takvim ayında en fazla 125 TL nakit iade kazanılabilir', "A101 Mağazalarında, a101.com.tr ya da Kapıda uygulamasında Hadi ürünleri ile yapacağın ödemelerde kampanyaya otomatik olarak katılır ve anında %3'e varan nakit iade kazanabilirsin. 1 Eylül 2026 itibarıyla iade oranı %1 olarak güncellenecektir."]

TOM Bank Hadi Taksitli Alışveriş Kredisi ile Samsung cep telefonlarında 3 Taksit Fırsatı!
Kategori       : Finansman Kampanyaları
Kâr Payı       : []
Finansman Oran

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import shutil
from urllib.parse import urlparse

from google.colab import files


V3_FILE = (
    "/content/"
    "tom_katilim_kampanya_extracted_v3.json"
)

FINAL_FILE = (
    "/content/"
    "tom_katilim_kampanya_extracted.json"
)


def get_slug(url):

    path = urlparse(
        str(url or "")
    ).path.rstrip("/")

    if not path:
        return ""

    return path.split("/")[-1]


def normalize(value):

    return (
        str(value or "")
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
        .strip()
    )


def contains(values, fragment):

    fragment = normalize(
        fragment
    )

    return any(
        fragment
        in normalize(value)
        for value in values
    )


# ============================================================
# LOAD
# ============================================================

with open(
    V3_FILE,
    "r",
    encoding="utf-8"
) as f:

    records = json.load(f)


by_slug = {
    get_slug(
        record["kaynak_url"]
    ):
    record
    for record in records
}


errors = []


# ============================================================
# 1. COUNT
# ============================================================

if len(records) != 53:

    errors.append(
        (
            "Kayıt sayısı yanlış: "
            f"{len(records)} != 53"
        )
    )


# ============================================================
# 2. DUPLICATE
# ============================================================

urls = [
    record["kaynak_url"]
    for record in records
]

duplicate_count = (
    len(urls)
    - len(set(urls))
)

if duplicate_count != 0:

    errors.append(
        (
            "Duplicate URL var: "
            f"{duplicate_count}"
        )
    )


# ============================================================
# 3. A101 %3
# ============================================================

slug = (
    "a101de-her-alisveriste-"
    "3-nakit-iade"
)

if slug not in by_slug:

    errors.append(
        "A101 %3 kampanyası bulunamadı."
    )

else:

    item = by_slug[slug]

    if (
        item["urun_kategorisi"]
        != "Kart Kampanyaları"
    ):

        errors.append(
            (
                "A101 kategori yanlış: "
                f"{item['urun_kategorisi']}"
            )
        )

    if not contains(
        item["hedef_kitle"],
        "Hadi Gold"
    ):

        errors.append(
            "A101 Hadi Gold hedef kitlesi yok."
        )

    if not contains(
        item["kampanya_avantaji"],
        "%3"
    ):

        errors.append(
            "A101 %3 avantajı yok."
        )


# ============================================================
# 4. GENEL SAMSUNG
#
# BURADA %0 BEKLEMİYORUZ.
# ============================================================

samsung_slug = (
    "tom-bank-hadi-taksitli-"
    "alisveris-kredisi-ile-"
    "samsung-cep-telefonlarinda-"
    "3-taksit-firsati"
)

if samsung_slug not in by_slug:

    errors.append(
        "Samsung kampanyası bulunamadı."
    )

else:

    item = by_slug[
        samsung_slug
    ]

    if (
        item["kar_payi_orani"]
        != []
    ):

        errors.append(
            (
                "Genel Samsung kampanyasında "
                "kâr payı boş olmalı: "
                f"{item['kar_payi_orani']}"
            )
        )

    if (
        item["taksit_sayisi"]
        != ["3"]
    ):

        errors.append(
            (
                "Samsung taksit yanlış: "
                f"{item['taksit_sayisi']}"
            )
        )


# ============================================================
# 5. ÇOK KAZANANLAR CEP TELEFONU
#
# %0 KÂR PAYI GERÇEKTE BU KAMPANYADA.
# ============================================================

club_phone_slug = (
    "tom-bank-cok-kazananlar-"
    "kulubune-ozel-vade-farksiz-"
    "3-taksit-cep-telefonu"
)

if club_phone_slug not in by_slug:

    errors.append(
        (
            "Çok Kazananlar Kulübü "
            "cep telefonu kampanyası bulunamadı."
        )
    )

else:

    item = by_slug[
        club_phone_slug
    ]

    if (
        item["urun_kategorisi"]
        != "Finansman Kampanyaları"
    ):

        errors.append(
            (
                "Çok Kazananlar cep telefonu "
                "kategori yanlış: "
                f"{item['urun_kategorisi']}"
            )
        )

    if not contains(
        item["kar_payi_orani"],
        "%0"
    ):

        errors.append(
            (
                "Çok Kazananlar cep telefonu -> "
                "%0 kâr payı bulunamadı: "
                f"{item['kar_payi_orani']}"
            )
        )

    if (
        item["taksit_sayisi"]
        != ["3"]
    ):

        errors.append(
            (
                "Çok Kazananlar cep telefonu -> "
                "taksit yanlış: "
                f"{item['taksit_sayisi']}"
            )
        )


# ============================================================
# 6. ECZANE
# ============================================================

eczane_slug = (
    "eczanelerde-hadi-saglik-"
    "kredisi-ile-0-vade-farki-"
    "ile-6-taksit"
)

if eczane_slug not in by_slug:

    errors.append(
        "Eczane kampanyası bulunamadı."
    )

else:

    item = by_slug[
        eczane_slug
    ]

    if not contains(
        item["finansman_orani"],
        "%0"
    ):

        errors.append(
            "Eczane %0 finansman oranı yok."
        )

    if (
        item["kar_payi_orani"]
        != []
    ):

        errors.append(
            (
                "Eczane kâr payı boş olmalı: "
                f"{item['kar_payi_orani']}"
            )
        )

    if (
        item["taksit_sayisi"]
        != ["3", "6"]
    ):

        errors.append(
            (
                "Eczane taksit yanlış: "
                f"{item['taksit_sayisi']}"
            )
        )


# ============================================================
# 7. ÖZEL BANKACILIK
# ============================================================

private_slugs = [
    (
        "restoran-harcamalarinda-"
        "her-ay-10-000-tlye-varan-iade"
    ),
    (
        "ucak-bileti-alimlarinda-"
        "ayda-3-500-tl-kazan"
    ),
]


for slug in private_slugs:

    if slug not in by_slug:

        errors.append(
            (
                "Özel Bankacılık kampanyası "
                f"bulunamadı: {slug}"
            )
        )

        continue

    item = by_slug[slug]

    if (
        item["urun_kategorisi"]
        != "Özel Bankacılık Kampanyaları"
    ):

        errors.append(
            (
                f"{slug} kategori yanlış: "
                f"{item['urun_kategorisi']}"
            )
        )


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

categories = {}

for record in records:

    category = (
        record[
            "urun_kategorisi"
        ]
    )

    categories[category] = (
        categories.get(
            category,
            0
        )
        + 1
    )


# ============================================================
# RESULT
# ============================================================

print(
    "=" * 110
)

print(
    "TOM KATILIM - "
    "CAMPAIGN FINAL AUDIT"
)

print(
    "=" * 110
)

print(
    "Kayıt           :",
    len(records)
)

print(
    "Duplicate URL   :",
    duplicate_count
)

print(
    "Validation error:",
    len(errors)
)


print()
print(
    "=" * 110
)

print(
    "KATEGORİ DAĞILIMI"
)

print(
    "=" * 110
)

for category, count in sorted(
    categories.items()
):

    print(
        f"{category}: {count}"
    )


# ============================================================
# RATE CHECK
# ============================================================

print()
print(
    "=" * 110
)

print(
    "ORAN KONTROLÜ"
)

print(
    "=" * 110
)


if samsung_slug in by_slug:

    print(
        "Genel Samsung kâr payı :",
        by_slug[
            samsung_slug
        ][
            "kar_payi_orani"
        ]
    )


if club_phone_slug in by_slug:

    print(
        "Kulüp Cep Telefonu kâr  :",
        by_slug[
            club_phone_slug
        ][
            "kar_payi_orani"
        ]
    )


if eczane_slug in by_slug:

    print(
        "Eczane finansman oranı  :",
        by_slug[
            eczane_slug
        ][
            "finansman_orani"
        ]
    )


# ============================================================
# ERRORS / FINAL
# ============================================================

print()
print(
    "=" * 110
)

if errors:

    print(
        "HATALAR"
    )

    print(
        "=" * 110
    )

    for error in errors:

        print(
            "-",
            error
        )

    print()
    print(
        "SONUÇ: TOM KATILIM "
        "KONTROL GEREKİYOR ❌"
    )

else:

    # V3 artık final dosyamız.
    shutil.copyfile(
        V3_FILE,
        FINAL_FILE
    )

    print(
        "SONUÇ: TOM KATILIM "
        "TAMAMEN BAŞARILI ✅"
    )

    print(
        "53 aktif kampanya "
        "final schema'da doğrulandı ✅"
    )

    print()
    print(
        "Final JSON:",
        FINAL_FILE
    )


print(
    "=" * 110
)


if not errors:

    files.download(
        FINAL_FILE
    )

TOM KATILIM - CAMPAIGN FINAL AUDIT
Kayıt           : 53
Duplicate URL   : 0
Validation error: 0

KATEGORİ DAĞILIMI
Alışveriş Kampanyaları: 2
Finansman Kampanyaları: 23
Kart Kampanyaları: 23
Sigorta Kampanyaları: 1
Yeni Müşteri Kampanyaları: 2
Özel Bankacılık Kampanyaları: 2

ORAN KONTROLÜ
Genel Samsung kâr payı : []
Kulüp Cep Telefonu kâr  : ['%0']
Eczane finansman oranı  : ['%0']

SONUÇ: TOM KATILIM TAMAMEN BAŞARILI ✅
53 aktif kampanya final schema'da doğrulandı ✅

Final JSON: /content/tom_katilim_kampanya_extracted.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
import re
from urllib.parse import urlparse

from google.colab import files


# ============================================================
# DOSYALAR
# ============================================================

FINANCE_RAW_FILE = (
    "/content/"
    "tom_katilim_finansmanlar.json"
)

CAMPAIGN_FILE = (
    "/content/"
    "tom_katilim_kampanya_extracted.json"
)

FINANCE_OUTPUT_FILE = (
    "/content/"
    "tom_katilim_finansman_extracted.json"
)

FINAL_OUTPUT_FILE = (
    "/content/"
    "tom_katilim_final.json"
)


BANK_NAME = "T.O.M. Katılım Bankası"


# ============================================================
# FINAL 18-KEY SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# NORMALIZATION
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    replacements = {
        "’": "'",
        "‘": "'",
        "–": "-",
        "—": "-",
        "\xa0": " ",
    }

    for old, new in replacements.items():

        value = value.replace(
            old,
            new
        )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    value = clean_text(
        value
    )

    value = value.replace(
        "İ",
        "i"
    )

    value = value.replace(
        "I",
        "ı"
    )

    return (
        value
        .casefold()
        .strip()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(
            value
        )

        if not value:
            continue

        key = normalize(
            value
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        result.append(
            value
        )

    return result


def get_slug(url):

    path = urlparse(
        str(url or "")
    ).path.rstrip("/")

    if not path:
        return ""

    return path.split("/")[-1]


def raw_lines(raw_text):

    result = []

    for line in str(
        raw_text or ""
    ).splitlines():

        line = clean_text(
            line
        )

        if line:

            result.append(
                line
            )

    return unique(
        result
    )


# ============================================================
# PRODUCT CATEGORY
# ============================================================

def finance_category(
    slug
):

    if slug == "veresiye-kredi":

        return (
            "Alışveriş Finansmanı"
        )

    if slug == (
        "taksitli-alisveris-kredisi"
    ):

        return (
            "Alışveriş ve Hizmet Finansmanı"
        )

    if slug == (
        "magazadan-alisveris-kredisi"
    ):

        return (
            "Mağazadan Alışveriş Finansmanı"
        )

    return "Finansman"


# ============================================================
# CONDITIONS
# ============================================================

def extract_conditions(
    slug,
    raw_text
):

    lines = raw_lines(
        raw_text
    )

    result = []

    # --------------------------------------------------------
    # VERESİYE
    # --------------------------------------------------------

    if slug == "veresiye-kredi":

        markers = (
            "2 ay sonra",
            "45 ila 75 gün",
            "veresiye limit",
            "borcunu ödemen gerekir",
            "otomatik olarak tahsil",
        )

    # --------------------------------------------------------
    # TAKSİTLİ
    # --------------------------------------------------------

    elif slug == (
        "taksitli-alisveris-kredisi"
    ):

        markers = (
            "taksitli kredi limit",
            "sana özel tanımlanmış limit",
            "borcunu ödemen gerekir",
            "otomatik olarak tahsil",
            "kredi skor",
        )

    # --------------------------------------------------------
    # MAĞAZADAN
    # --------------------------------------------------------

    elif slug == (
        "magazadan-alisveris-kredisi"
    ):

        markers = (
            "1.000 tl",
            "200.000 tl",
            "36 aya varan",
            "bankamız müşterisi",
            "kredi kullanmak için",
            "anlaşmalı mağaza",
        )

    else:

        markers = (
            "gerekm",
            "geçerli",
            "limit",
            "ödeme",
        )

    for line in lines:

        normalized = normalize(
            line
        )

        if any(
            marker
            in normalized
            for marker
            in markers
        ):

            result.append(
                line
            )

    return unique(
        result
    )


# ============================================================
# FINANCE RECORD
# ============================================================

def finance_record(
    raw
):

    title = clean_text(
        raw.get(
            "urun_adi",
            ""
        )
    )

    url = clean_text(
        raw.get(
            "kaynak_url",
            ""
        )
    )

    raw_text = clean_text(
        raw.get(
            "ham_metin",
            ""
        )
    )

    slug = get_slug(
        url
    )

    # --------------------------------------------------------
    # DEFAULTS
    # --------------------------------------------------------

    kar_payi = []

    finansman_orani = []

    finansman_tutari = []

    vade = []

    taksit = []

    masraf = []

    hedef_kitle = []

    para_birimi = []


    # ========================================================
    # VERESİYE
    # ========================================================

    if slug == "veresiye-kredi":

        # Kaynak iki şekilde ifade ediyor:
        # 2 ay + harcamaya göre 45-75 gün.
        vade = [
            "2 ay",
            "45 ila 75 gün",
        ]


    # ========================================================
    # TAKSİTLİ KREDİ
    # ========================================================

    elif slug == (
        "taksitli-alisveris-kredisi"
    ):

        # Sabit oran / sabit taksit / sabit limit
        # ürün sayfasında yayınlanmıyor.
        kar_payi = []
        finansman_orani = []
        finansman_tutari = []
        vade = []
        taksit = []


    # ========================================================
    # MAĞAZADAN ALIŞVERİŞ KREDİSİ
    # ========================================================

    elif slug == (
        "magazadan-alisveris-kredisi"
    ):

        finansman_tutari = [
            "1.000 TL - 200.000 TL"
        ]

        vade = [
            "36 aya varan"
        ]

        taksit = [
            "36"
        ]

        para_birimi = [
            "TL"
        ]

        hedef_kitle = [
            (
                "Anlaşmalı mağazalardan "
                "alışveriş yapan müşteriler"
            )
        ]


    # ========================================================
    # RAW'DAKİ EXPLICIT ORANLARI KORU
    # ========================================================

    if raw.get(
        "kar_payi_orani"
    ):

        kar_payi = unique(
            raw[
                "kar_payi_orani"
            ]
        )

    if raw.get(
        "finansman_orani"
    ):

        finansman_orani = unique(
            raw[
                "finansman_orani"
            ]
        )


    # ========================================================
    # MASRAF
    # ========================================================

    for value in raw.get(
        "masraf_bilgisi",
        []
    ):

        normalized = normalize(
            value
        )

        if any(
            x in normalized
            for x in (
                "masraf",
                "komisyon",
                "tahsis ücreti",
                "ücret alın",
            )
        ):

            masraf.append(
                value
            )

    masraf = unique(
        masraf
    )


    # ========================================================
    # FINAL RECORD
    # ========================================================

    return {

        "banka":
            BANK_NAME,

        "kayit_turu":
            "finansman",

        "urun_adi":
            title,

        "urun_kategorisi":
            finance_category(
                slug
            ),

        "kar_payi_orani":
            kar_payi,

        "finansman_orani":
            finansman_orani,

        "finansman_tutari":
            finansman_tutari,

        "vade":
            vade,

        "taksit_sayisi":
            taksit,

        "masraf_bilgisi":
            masraf,

        "kampanya_turu":
            "",

        "kampanya_avantaji":
            [],

        "kampanya_suresi":
            "",

        "hedef_kitle":
            hedef_kitle,

        "para_birimi":
            para_birimi,

        "kosullar":
            extract_conditions(
                slug,
                raw_text
            ),

        "kaynak_url":
            url,

        "ham_metin":
            raw_text,
    }


# ============================================================
# SCHEMA VALIDATION
# ============================================================

def validate_schema(
    record,
    index
):

    errors = []

    if (
        list(
            record.keys()
        )
        != SCHEMA_KEYS
    ):

        errors.append(
            (
                f"[{index}] "
                "key/order uyuşmuyor."
            )
        )

    for field in LIST_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            list
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} list değil."
                )
            )

    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(
                field
            ),
            str
        ):

            errors.append(
                (
                    f"[{index}] "
                    f"{field} string değil."
                )
            )

    if not record.get(
        "urun_adi"
    ):

        errors.append(
            (
                f"[{index}] "
                "urun_adi boş."
            )
        )

    if not record.get(
        "kaynak_url"
    ):

        errors.append(
            (
                f"[{index}] "
                "kaynak_url boş."
            )
        )

    if not record.get(
        "ham_metin"
    ):

        errors.append(
            (
                f"[{index}] "
                "ham_metin boş."
            )
        )

    return errors


# ============================================================
# LOAD FINANCE RAW
# ============================================================

with open(
    FINANCE_RAW_FILE,
    "r",
    encoding="utf-8"
) as f:

    finance_raw = json.load(
        f
    )


if not isinstance(
    finance_raw,
    list
):

    raise TypeError(
        (
            "tom_katilim_finansmanlar.json "
            "liste formatında değil."
        )
    )


# ============================================================
# FINANCE EXTRACTION
# ============================================================

finance_records = []

finance_errors = []


for index, raw in enumerate(
    finance_raw,
    start=1
):

    try:

        record = finance_record(
            raw
        )

        finance_records.append(
            record
        )

        finance_errors.extend(
            validate_schema(
                record,
                index
            )
        )

    except Exception as e:

        finance_errors.append(
            (
                f"[{index}] "
                f"{type(e).__name__}: "
                f"{e}"
            )
        )


# ============================================================
# FINANCE SEMANTIC AUDIT
# ============================================================

by_slug = {
    get_slug(
        record[
            "kaynak_url"
        ]
    ):
    record
    for record
    in finance_records
}


expected_slugs = {
    "veresiye-kredi",
    "taksitli-alisveris-kredisi",
    "magazadan-alisveris-kredisi",
}


if set(
    by_slug.keys()
) != expected_slugs:

    finance_errors.append(
        (
            "Finansman ürün URL'leri "
            "beklenen 3 ürünle uyuşmuyor: "
            f"{set(by_slug.keys())}"
        )
    )


# ------------------------------------------------------------
# VERESİYE
# ------------------------------------------------------------

if "veresiye-kredi" in by_slug:

    item = by_slug[
        "veresiye-kredi"
    ]

    if "2 ay" not in item[
        "vade"
    ]:

        finance_errors.append(
            (
                "Veresiye -> "
                "2 ay vade yok."
            )
        )

    if (
        "45 ila 75 gün"
        not in item[
            "vade"
        ]
    ):

        finance_errors.append(
            (
                "Veresiye -> "
                "45 ila 75 gün yok."
            )
        )


# ------------------------------------------------------------
# TAKSİTLİ KREDİ
# ------------------------------------------------------------

if (
    "taksitli-alisveris-kredisi"
    in by_slug
):

    item = by_slug[
        "taksitli-alisveris-kredisi"
    ]

    # Ürün sayfasında sabit taksit / oran
    # yayınlanmadığından boş olmasını bekliyoruz.

    if item[
        "kar_payi_orani"
    ]:

        finance_errors.append(
            (
                "Taksitli Kredi -> "
                "sabit kâr payı "
                "olmaması bekleniyor: "
                f"{item['kar_payi_orani']}"
            )
        )


# ------------------------------------------------------------
# MAĞAZADAN
# ------------------------------------------------------------

if (
    "magazadan-alisveris-kredisi"
    in by_slug
):

    item = by_slug[
        "magazadan-alisveris-kredisi"
    ]

    if (
        item[
            "finansman_tutari"
        ]
        != [
            "1.000 TL - 200.000 TL"
        ]
    ):

        finance_errors.append(
            (
                "Mağazadan -> tutar yanlış: "
                f"{item['finansman_tutari']}"
            )
        )

    if (
        item[
            "vade"
        ]
        != [
            "36 aya varan"
        ]
    ):

        finance_errors.append(
            (
                "Mağazadan -> vade yanlış: "
                f"{item['vade']}"
            )
        )

    if (
        item[
            "taksit_sayisi"
        ]
        != [
            "36"
        ]
    ):

        finance_errors.append(
            (
                "Mağazadan -> taksit yanlış: "
                f"{item['taksit_sayisi']}"
            )
        )


# ============================================================
# SAVE FINANCE
# ============================================================

with open(
    FINANCE_OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        finance_records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# LOAD FINAL CAMPAIGNS
# ============================================================

with open(
    CAMPAIGN_FILE,
    "r",
    encoding="utf-8"
) as f:

    campaign_records = json.load(
        f
    )


if not isinstance(
    campaign_records,
    list
):

    raise TypeError(
        "Campaign final JSON list değil."
    )


# ============================================================
# COMBINE
# ============================================================

final_records = (
    finance_records
    +
    campaign_records
)


# ============================================================
# FINAL AUDIT
# ============================================================

final_errors = []


# ------------------------------------------------------------
# COUNT
# ------------------------------------------------------------

if len(
    finance_records
) != 3:

    final_errors.append(
        (
            "Finansman sayısı "
            f"3 değil: {len(finance_records)}"
        )
    )


if len(
    campaign_records
) != 53:

    final_errors.append(
        (
            "Kampanya sayısı "
            f"53 değil: {len(campaign_records)}"
        )
    )


if len(
    final_records
) != 56:

    final_errors.append(
        (
            "Final kayıt sayısı "
            f"56 değil: {len(final_records)}"
        )
    )


# ------------------------------------------------------------
# SCHEMA
# ------------------------------------------------------------

for index, record in enumerate(
    final_records,
    start=1
):

    final_errors.extend(
        validate_schema(
            record,
            index
        )
    )


# ------------------------------------------------------------
# BANK
# ------------------------------------------------------------

wrong_bank = [
    record
    for record
    in final_records
    if (
        record[
            "banka"
        ]
        != BANK_NAME
    )
]


if wrong_bank:

    final_errors.append(
        (
            "Banka adı farklı "
            f"{len(wrong_bank)} kayıt var."
        )
    )


# ------------------------------------------------------------
# RECORD TYPES
# ------------------------------------------------------------

finance_count = sum(
    1
    for record
    in final_records
    if (
        record[
            "kayit_turu"
        ]
        == "finansman"
    )
)


campaign_count = sum(
    1
    for record
    in final_records
    if (
        record[
            "kayit_turu"
        ]
        == "kampanya"
    )
)


if finance_count != 3:

    final_errors.append(
        (
            "Final finansman sayısı "
            f"3 değil: {finance_count}"
        )
    )


if campaign_count != 53:

    final_errors.append(
        (
            "Final kampanya sayısı "
            f"53 değil: {campaign_count}"
        )
    )


# ------------------------------------------------------------
# URL DUPLICATE
# ------------------------------------------------------------

urls = [
    record[
        "kaynak_url"
    ]
    for record
    in final_records
]


duplicate_urls = (
    len(urls)
    -
    len(
        set(urls)
    )
)


if duplicate_urls:

    final_errors.append(
        (
            "Final duplicate URL: "
            f"{duplicate_urls}"
        )
    )


# ============================================================
# SAVE FINAL
# ============================================================

with open(
    FINAL_OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# OUTPUT
# ============================================================

print(
    "=" * 110
)

print(
    "TOM KATILIM - "
    "FINAL BANK AUDIT"
)

print(
    "=" * 110
)

print(
    "Finansman RAW       :",
    len(finance_raw)
)

print(
    "Finansman extracted :",
    len(finance_records)
)

print(
    "Finansman error     :",
    len(finance_errors)
)

print(
    "Kampanya final      :",
    len(campaign_records)
)

print(
    "Toplam final        :",
    len(final_records)
)

print(
    "Final finansman     :",
    finance_count
)

print(
    "Final kampanya      :",
    campaign_count
)

print(
    "Duplicate URL       :",
    duplicate_urls
)

print(
    "Final audit error   :",
    len(final_errors)
)


# ============================================================
# FINANCE RECORDS
# ============================================================

print()
print(
    "=" * 110
)

print(
    "FİNANSMAN ÜRÜNLERİ"
)

print(
    "=" * 110
)


for record in finance_records:

    print()
    print(
        record[
            "urun_adi"
        ]
    )

    print(
        "Kategori :",
        record[
            "urun_kategorisi"
        ]
    )

    print(
        "Tutar    :",
        record[
            "finansman_tutari"
        ]
    )

    print(
        "Vade     :",
        record[
            "vade"
        ]
    )

    print(
        "Taksit   :",
        record[
            "taksit_sayisi"
        ]
    )

    print(
        "Kâr Payı :",
        record[
            "kar_payi_orani"
        ]
    )


# ============================================================
# ERRORS
# ============================================================

all_errors = (
    finance_errors
    +
    final_errors
)


if all_errors:

    print()
    print(
        "=" * 110
    )

    print(
        "HATALAR"
    )

    print(
        "=" * 110
    )

    for error in all_errors:

        print(
            "-",
            error
        )


# ============================================================
# FINAL RESULT
# ============================================================

print()
print(
    "=" * 110
)


if not all_errors:

    print(
        "SONUÇ: TOM KATILIM "
        "TAMAMEN KAPANDI ✅"
    )

    print(
        "3 finansman + "
        "53 kampanya = "
        "56 final kayıt ✅"
    )

    print()

    print(
        "Finansman JSON:",
        FINANCE_OUTPUT_FILE
    )

    print(
        "Final Bank JSON:",
        FINAL_OUTPUT_FILE
    )

else:

    print(
        "SONUÇ: TOM KATILIM "
        "KONTROL GEREKİYOR ❌"
    )


print(
    "=" * 110
)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    FINANCE_OUTPUT_FILE
)

files.download(
    FINAL_OUTPUT_FILE
)

TOM KATILIM - FINAL BANK AUDIT
Finansman RAW       : 3
Finansman extracted : 3
Finansman error     : 0
Kampanya final      : 53
Toplam final        : 56
Final finansman     : 3
Final kampanya      : 53
Duplicate URL       : 0
Final audit error   : 0

FİNANSMAN ÜRÜNLERİ

Veresiye Kredi
Kategori : Alışveriş Finansmanı
Tutar    : []
Vade     : ['2 ay', '45 ila 75 gün']
Taksit   : []
Kâr Payı : []

Taksitli Kredi
Kategori : Alışveriş ve Hizmet Finansmanı
Tutar    : []
Vade     : []
Taksit   : []
Kâr Payı : []

Mağazadan Alışveriş Kredisi
Kategori : Mağazadan Alışveriş Finansmanı
Tutar    : ['1.000 TL - 200.000 TL']
Vade     : ['36 aya varan']
Taksit   : ['36']
Kâr Payı : []

SONUÇ: TOM KATILIM TAMAMEN KAPANDI ✅
3 finansman + 53 kampanya = 56 final kayıt ✅

Finansman JSON: /content/tom_katilim_finansman_extracted.json
Final Bank JSON: /content/tom_katilim_final.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>